# LeetCode 308 — Range Sum Query 2D - Mutable (Hard): từ prefix sum đến Fenwick Tree 2D bằng Python

> **Lưu ý:** đây là bài **Premium** (bị khóa trên LeetCode, cần tài khoản trả phí mới xem đề và nộp bài). Notebook ghi lại đề bài, ví dụ, constraints và signature Python3 theo bản LeetCode, đồng thời tự viết hàm `run_commands` mô phỏng cách trình chấm gọi class — nên bạn vẫn học và chạy thử đầy đủ mà không cần Premium.

Skeleton trên LeetCode (Python3):

```python
class NumMatrix:

    def __init__(self, matrix: List[List[int]]):


    def update(self, row: int, col: int, val: int) -> None:


    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:



# Your NumMatrix object will be instantiated and called as such:
# obj = NumMatrix(matrix)
# obj.update(row,col,val)
# param_2 = obj.sumRegion(row1,col1,row2,col2)
```

Đây là bài **thiết kế class** (design): ta không viết một hàm `Solution.xxx` chạy một lần, mà viết cả một class `NumMatrix` có **trạng thái**, và hai loại thao tác bị gọi **xen kẽ nhau nhiều lần**:

- `update` — **sửa** giá trị một ô;
- `sumRegion` — **hỏi tổng** một hình chữ nhật.

Cái khó nằm ở chỗ: cấu trúc nào làm việc **hỏi** thật nhanh thì thường làm việc **sửa** rất chậm, và ngược lại. Bài này dạy ta cách **cân bằng** hai loại chi phí đó.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách LeetCode chấm bài thiết kế class (`["NumMatrix","sumRegion",...]` và `[[...],[2,1,4,3],...]`) và tự viết hàm `run_commands` để chạy y hệt.
2. Ôn từ căn bản 3 bài nền: **#303** (prefix sum 1D), **#304** (prefix sum 2D, ma trận không đổi), **#307** (mảng 1D có update).
3. Bốn bậc lời giải, mỗi bậc sửa đúng chỗ vướng của bậc trước:
   - Cách 1 — **Ngây thơ**: update `O(1)`, sumRegion cộng từng ô `O(mn)`.
   - Cách 2 — **Prefix sum 2D** của #304: sumRegion `O(1)` nhưng update `O(mn)`.
   - Cách 3 — **Prefix sum theo từng hàng**: update `O(n)`, sumRegion `O(m)` — cân bằng hơn.
   - Cách 4 (cách chính) — **Fenwick Tree 2D**: cả hai thao tác `O(log m · log n)`.
4. **Fenwick Tree (Binary Indexed Tree) 1D dạy từ con số 0**: `lowbit = i & -i`, "cây trách nhiệm", vì sao hỏi thì nhảy `i -= lowbit(i)` còn sửa thì nhảy `i += lowbit(i)`.
5. Hai hướng mở rộng (cách 5): mỗi hàng một Fenwick 1D, và ý tưởng **segment tree 2D**.
6. Cái bẫy quan trọng nhất của bài: `update` là **GÁN** giá trị mới, không phải cộng thêm → phải tính `delta = val - giá_trị_cũ`.
7. Trace từng bước, test ví dụ đề + edge case, **đối chiếu chéo 6 cài đặt** trên chuỗi lệnh ngẫu nhiên có `random.seed`, lỗi hay gặp, 6 bài tập có gợi ý và đáp án ở cuối.

## 1. Đề bài

Cho một ma trận số nguyên 2 chiều `matrix`. Hãy xử lý **nhiều truy vấn** thuộc hai loại sau:

1. **Cập nhật** (update) giá trị của một ô trong `matrix`.
2. **Tính tổng** các phần tử của `matrix` nằm trong hình chữ nhật xác định bởi góc **trên-trái** `(row1, col1)` và góc **dưới-phải** `(row2, col2)`.

Cài đặt class `NumMatrix`:

- `NumMatrix(int[][] matrix)` — khởi tạo đối tượng với ma trận số nguyên `matrix`.
- `void update(int row, int col, int val)` — **cập nhật** giá trị của `matrix[row][col]` thành `val`.
- `int sumRegion(int row1, int col1, int row2, int col2)` — trả về tổng các phần tử của `matrix` nằm trong hình chữ nhật có góc trên-trái `(row1, col1)` và góc dưới-phải `(row2, col2)` (tính cả các ô nằm trên cạnh).

**Example 1**

```text
Input
["NumMatrix", "sumRegion", "update", "sumRegion"]
[[[[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]]], [2, 1, 4, 3], [3, 2, 2], [2, 1, 4, 3]]
Output
[null, 8, null, 10]

Explanation
NumMatrix numMatrix = new NumMatrix([[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]]);
numMatrix.sumRegion(2, 1, 4, 3); // return 8 (i.e. sum of the left red rectangle)
numMatrix.update(3, 2, 2);       // matrix changes from left image to right image
numMatrix.sumRegion(2, 1, 4, 3); // return 10 (i.e. sum of the right red rectangle)
```

(Trên LeetCode có hình: hình chữ nhật đỏ bên trái là trước khi update, bên phải là sau update. Notebook sẽ "vẽ" lại bằng chữ ngay bên dưới.)

**Constraints**

- `m == matrix.length`
- `n == matrix[i].length`
- `1 <= m, n <= 200`
- `-1000 <= matrix[i][j] <= 1000`
- `0 <= row < m`
- `0 <= col < n`
- `-1000 <= val <= 1000`
- `0 <= row1 <= row2 < m`
- `0 <= col1 <= col2 < n`
- At most `5000` calls will be made to `sumRegion` and `update` (tổng số lần gọi hai hàm này **tối đa 5000**).

Ghi chú: các bản đề cũ có thêm câu *"You may assume the number of calls to update and sumRegion function is distributed evenly"* — số lần gọi `update` và `sumRegion` **phân bố đều nhau**. Câu này là gợi ý quan trọng: ta **không được** hy sinh hẳn một loại thao tác để loại kia thật nhanh.

**Quy ước dùng trong notebook**

- `m` = số hàng, `n` = số cột (`m, n <= 200` nên `m·n <= 40 000`);
- `Q` = tổng số lần gọi `update` + `sumRegion` (`Q <= 5000`);
- `log` là log cơ số 2 (`log2(200) ≈ 7.6`);
- ô `(r, c)` nghĩa là hàng `r`, cột `c`, **đếm từ 0** như Python — trừ khi ghi rõ "1-index".

### Đọc format test của bài thiết kế class

Với bài thiết kế class, LeetCode cho **hai danh sách song song**: danh sách tên lệnh và danh sách đối số.

| Vị trí | Tên lệnh (`ops`) | Đối số (`args`) | Lời gọi Python tương ứng | Kết quả |
|---:|---|---|---|---|
| 0 | `"NumMatrix"` | `[matrix]` | `obj = NumMatrix(matrix)` | `null` (hàm khởi tạo không trả gì) |
| 1 | `"sumRegion"` | `[2, 1, 4, 3]` | `obj.sumRegion(2, 1, 4, 3)` | `8` |
| 2 | `"update"` | `[3, 2, 2]` | `obj.update(3, 2, 2)` | `null` (update trả `None`) |
| 3 | `"sumRegion"` | `[2, 1, 4, 3]` | `obj.sumRegion(2, 1, 4, 3)` | `10` |

`null` trong JSON chính là `None` trong Python.

Một điểm rất dễ hiểu nhầm: `update(3, 2, 2)` nghĩa là **đặt** `matrix[3][2] = 2` (ô đó trước đây là `0`), **không phải** "cộng thêm 2". Ở ví dụ này hai cách hiểu tình cờ cho cùng kết quả (vì `0 + 2 = 2`), nên ví dụ của đề **không bắt được** lỗi hiểu nhầm — ta sẽ tự thêm test để bắt.

Cell dưới "vẽ" ma trận, đóng ngoặc `[ ]` các ô nằm trong hình chữ nhật `(2,1) → (4,3)`: hàng 2..4, cột 1..3. Hàm `copy.deepcopy` tạo một bản sao **hoàn toàn mới** của list lồng nhau, để ta sửa bản sao mà ma trận gốc `EXAMPLE_MATRIX` vẫn nguyên.

In [1]:
import copy

EXAMPLE_MATRIX = [
    [3, 0, 1, 4, 2],
    [5, 6, 3, 2, 1],
    [1, 2, 0, 1, 5],
    [4, 1, 0, 1, 7],
    [1, 0, 3, 0, 5],
]

def show_region(matrix, row1, col1, row2, col2):
    # In ma trận; ô nằm trong hình chữ nhật được đóng ngoặc [ ]. Trả về tổng vùng.
    total = 0
    print("      " + "".join(f" c{c}  " for c in range(len(matrix[0]))))
    for r, row in enumerate(matrix):
        parts = []
        for c, v in enumerate(row):
            if row1 <= r <= row2 and col1 <= c <= col2:
                parts.append(f"[{v:>2}]")
                total += v
            else:
                parts.append(f" {v:>2} ")
        print(f"r{r}:   " + " ".join(parts))
    print(f"Tổng vùng ({row1},{col1}) -> ({row2},{col2}) = {total}")
    return total

m1 = copy.deepcopy(EXAMPLE_MATRIX)
print("TRƯỚC update:")
show_region(m1, 2, 1, 4, 3)
m1[3][2] = 2                      # update(3, 2, 2): GÁN ô (3, 2) thành 2
print("\nSAU update(3, 2, 2):")
show_region(m1, 2, 1, 4, 3)
print("\nMa trận gốc vẫn nguyên? ", EXAMPLE_MATRIX[3][2] == 0)

TRƯỚC update:
       c0   c1   c2   c3   c4  
r0:     3    0    1    4    2 
r1:     5    6    3    2    1 
r2:     1  [ 2] [ 0] [ 1]   5 
r3:     4  [ 1] [ 0] [ 1]   7 
r4:     1  [ 0] [ 3] [ 0]   5 
Tổng vùng (2,1) -> (4,3) = 8

SAU update(3, 2, 2):
       c0   c1   c2   c3   c4  
r0:     3    0    1    4    2 
r1:     5    6    3    2    1 
r2:     1  [ 2] [ 0] [ 1]   5 
r3:     4  [ 1] [ 2] [ 1]   7 
r4:     1  [ 0] [ 3] [ 0]   5 
Tổng vùng (2,1) -> (4,3) = 10

Ma trận gốc vẫn nguyên?  True


## 2. Nền tảng 1 — Class giữ trạng thái, gọi method theo tên

### 2.1. Vì sao cần class?

Ở bài bình thường, LeetCode gọi **một** hàm **một** lần. Ở bài này, đối tượng `obj` được tạo **một lần**, rồi bị gọi `update`/`sumRegion` **nhiều lần**. Mọi thứ ta chuẩn bị trong `__init__` (bản sao ma trận, bảng prefix, cây Fenwick...) phải được **cất vào `self`** để các lần gọi sau dùng lại:

- `self.x = ...` trong `__init__`: tạo "ngăn kéo" tên `x` gắn với **đối tượng** đó.
- Trong `update`/`sumRegion`: đọc/sửa `self.x` — thay đổi được giữ lại cho lần gọi sau.
- Một method không có `return` thì Python tự trả về `None` — đúng với `null` của `update`.

### 2.2. Gọi method khi chỉ biết **tên** dạng chuỗi

Trình chấm chỉ có chuỗi `"update"`. Hàm có sẵn `getattr(obj, "update")` trả về **chính method** `obj.update`; thêm `(...)` là gọi được.

### 2.3. Dấu `*` "mở gói" một list thành nhiều đối số

`f(*[3, 2, 2])` giống hệt `f(3, 2, 2)`. Nhờ vậy ta truyền thẳng `args[k]` của đề vào method mà không cần biết method có mấy tham số.

Thử với một class đồ chơi `Tally` (bộ đếm tổng):

In [2]:
class Tally:
    def __init__(self, start):
        self.total = start          # trạng thái được giữ giữa các lần gọi

    def add(self, x):
        self.total += x             # không có return -> Python tự trả None

    def get(self):
        return self.total

t = Tally(10)
t.add(5)
print("get() =", t.get())                 # 15

method = getattr(t, "add")                # lấy method theo TÊN (chuỗi)
method(7)                                 # giống hệt t.add(7)
print("get() =", getattr(t, "get")())     # 22

args = [3]
t.add(*args)                              # *args mở gói list: t.add(3)
print("get() =", t.get())                 # 25
print("add(...) trả về:", t.add(1))       # None

get() = 15
get() = 22
get() = 25
add(...) trả về: None


### 2.4. Ghép lại: hàm `run_commands` mô phỏng trình chấm

```text
obj = cls(*args[0])                    # lệnh đầu tiên luôn là hàm khởi tạo -> kết quả None
với mỗi lệnh tiếp theo (op, a):
    kết quả = getattr(obj, op)(*a)     # gọi method theo tên, mở gói đối số
```

Hai chi tiết nhỏ nhưng quan trọng:

- `copy.deepcopy(args[0])`: mỗi lần chạy, đối tượng nhận một **bản sao mới** của ma trận đề (giống LeetCode tạo dữ liệu mới cho mỗi test). Nhờ vậy, dù một cài đặt nào đó lỡ sửa trực tiếp ma trận nó nhận được, các test sau vẫn không bị hỏng.
- Ta tham số hóa `cls` (class cần chạy) để dùng **cùng một hàm** cho mọi cài đặt: `NumMatrixNaive`, `NumMatrixRowPrefix`, `NumMatrix`... và cả bài #303, #307.

In [3]:
def run_commands(cls, ops, args):
    # Mô phỏng trình chấm LeetCode cho bài thiết kế class.
    obj = cls(*copy.deepcopy(args[0]))        # ops[0] là tên class -> tạo đối tượng
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))      # gọi method theo tên
    return out

print(run_commands(Tally, ["Tally", "add", "get", "add", "get"], [[10], [5], [], [-3], []]))

# Example 1 của đề — dùng lại cho mọi cài đặt phía sau
EX1_OPS = ["NumMatrix", "sumRegion", "update", "sumRegion"]
EX1_ARGS = [[EXAMPLE_MATRIX], [2, 1, 4, 3], [3, 2, 2], [2, 1, 4, 3]]
EX1_EXPECTED = [None, 8, None, 10]

[None, None, 15, None, 12]


## 3. Nền tảng 2 — Ôn #303: prefix sum 1D (mảng không đổi)

Bài **#303 Range Sum Query - Immutable**: cho mảng `nums`, nhiều lần hỏi `sumRange(left, right)` = tổng `nums[left..right]` (tính cả hai đầu). Mảng **không bao giờ đổi**.

Cách ngây thơ: mỗi lần hỏi cộng lại từ `left` tới `right` → `O(n)` mỗi lần, hỏi nhiều lần thì cộng đi cộng lại cùng những phần tử.

**Prefix sum** (tổng tiền tố): tính sẵn **một lần** mảng `P`:

```text
P[0] = 0
P[k] = nums[0] + nums[1] + ... + nums[k-1]      (tổng k phần tử ĐẦU TIÊN)
     = P[k-1] + nums[k-1]                       (tiền tố mới = tiền tố cũ + phần tử mới)
```

Ví dụ `nums = [-2, 0, 3, -5, 2, -1]` (Example của #303):

| k | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---:|---:|---:|---:|---:|---:|---:|
| phần tử vừa cộng `nums[k-1]` | – | -2 | 0 | 3 | -5 | 2 | -1 |
| `P[k]` | 0 | -2 | -2 | 1 | -4 | -2 | -3 |

Tổng một đoạn = **hiệu hai tiền tố** (tiền tố dài trừ tiền tố ngắn, phần chung triệt tiêu):

```text
sum(nums[left..right]) = P[right + 1] - P[left]
```

Ví dụ `sumRange(2, 5)` = `P[6] - P[2]` = `-3 - (-2)` = `-1`, đúng bằng `3 + (-5) + 2 + (-1)`.

**Vì sao `P` dài `n + 1` và có `P[0] = 0`?** Để công thức đúng cả khi `left = 0`: `sumRange(0, 2) = P[3] - P[0] = 1 - 0 = 1`, không cần `if` riêng. Mẹo "thêm một số 0 ở đầu" này sẽ còn gặp lại ở 2D (thêm một hàng 0 và một cột 0) và ở Fenwick (chỉ số bắt đầu từ 1).

Cell dưới tính `P`, rồi đóng gói thành class `NumArray` của #303 và chạy bằng `run_commands`.

In [4]:
nums = [-2, 0, 3, -5, 2, -1]
P = [0]
for x in nums:
    P.append(P[-1] + x)          # P[-1] là phần tử cuối cùng hiện có của P
print("P =", P)
print("sumRange(2, 5) = P[6] - P[2] =", P[6] - P[2])

class NumArrayImmutable:             # lời giải #303
    def __init__(self, nums):
        self.P = [0]
        for x in nums:
            self.P.append(self.P[-1] + x)

    def sumRange(self, left, right):
        return self.P[right + 1] - self.P[left]

print(run_commands(NumArrayImmutable,
                   ["NumArray", "sumRange", "sumRange", "sumRange"],
                   [[[-2, 0, 3, -5, 2, -1]], [0, 2], [2, 5], [0, 5]]))   # đề #303: [null, 1, -1, -3]

P = [0, -2, -2, 1, -4, -2, -3]
sumRange(2, 5) = P[6] - P[2] = -1
[None, 1, -1, -3]


## 4. Nền tảng 3 — Ôn #304: prefix sum 2D (ma trận không đổi)

Bài **#304 Range Sum Query 2D - Immutable** chính là bài của ta nhưng **không có `update`**. Lời giải kinh điển: mở rộng prefix sum lên 2 chiều.

### 4.1. Định nghĩa

Tạo bảng `P` kích thước `(m + 1) × (n + 1)`:

```text
P[i][j] = tổng mọi ô matrix[r][c] với 0 <= r < i và 0 <= c < j
        = tổng hình chữ nhật "i hàng đầu × j cột đầu" (neo ở góc trên-trái)
```

Hàng `P[0][*]` và cột `P[*][0]` toàn số 0 (giống `P[0] = 0` ở 1D).

### 4.2. Xây bảng bằng bao hàm – loại trừ (inclusion–exclusion)

```text
P[i][j] = matrix[i-1][j-1] + P[i-1][j] + P[i][j-1] - P[i-1][j-1]
```

Hình dung hình chữ nhật `i × j` chia làm 4 mảnh:

```text
+-----------+---+
|           |   |     P[i-1][j]   = D + B   (phần TRÊN: i-1 hàng đầu, đủ j cột)
|     D     | B |     P[i][j-1]   = D + C   (phần TRÁI: đủ i hàng, j-1 cột đầu)
|           |   |     => D bị cộng 2 lần, nên trừ P[i-1][j-1] = D một lần
+-----------+---+
|     C     | x |     x = matrix[i-1][j-1]: ô góc dưới-phải, chưa ai cộng
+-----------+---+
```

`(D + B) + (D + C) - D + x = D + B + C + x` = đúng cả hình chữ nhật.

Ví dụ với ma trận của đề: `P[2][2] = matrix[1][1] + P[1][2] + P[2][1] - P[1][1] = 6 + 3 + 8 - 3 = 14`, đúng bằng `3 + 0 + 5 + 6`.

Chú ý cách tạo bảng 2D: `[[0] * (n + 1) for _ in range(m + 1)]` — mỗi vòng `for` tạo **một list mới** cho từng hàng (mục Lỗi 4 sẽ cho thấy vì sao không được viết `[[0] * (n + 1)] * (m + 1)`).

In [5]:
def build_prefix_2d(matrix):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]     # (m+1) x (n+1), hàng 0 và cột 0 toàn 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            P[i][j] = matrix[i-1][j-1] + P[i-1][j] + P[i][j-1] - P[i-1][j-1]
    return P

P2 = build_prefix_2d(EXAMPLE_MATRIX)
for i, row in enumerate(P2):
    print(f"P[{i}] =", " ".join(f"{v:>3}" for v in row))
print("Kiểm tra P[2][2] =", P2[2][2], "| tự cộng 3 + 0 + 5 + 6 =", 3 + 0 + 5 + 6)

P[0] =   0   0   0   0   0   0
P[1] =   0   3   3   4   8  10
P[2] =   0   8  14  18  24  27
P[3] =   0   9  17  21  28  36
P[4] =   0  13  22  26  34  49
P[5] =   0  14  23  30  38  58
Kiểm tra P[2][2] = 14 | tự cộng 3 + 0 + 5 + 6 = 14


### 4.3. Truy vấn một hình chữ nhật bất kỳ trong `O(1)`

Muốn tổng vùng hàng `row1..row2`, cột `col1..col2`:

```text
sumRegion = P[row2+1][col2+1] - P[row1][col2+1] - P[row2+1][col1] + P[row1][col1]
                  A                   B                  C                 D
```

```text
                col1         col2
        +--------+------------+
        |   D    |            |      A = P[row2+1][col2+1] : khối từ (0,0) tới HẾT vùng
        |        |            |      B = P[row1][col2+1]   : mọi hàng PHÍA TRÊN row1 (gồm cả D)
  row1  +--------+------------+      C = P[row2+1][col1]   : mọi cột BÊN TRÁI col1 (gồm cả D)
        |        |  VÙNG CẦN  |      D = P[row1][col1]     : góc trên-trái, bị trừ 2 lần
        |   C    |    TÍNH    |                              (trong B và trong C) -> cộng trả 1 lần
  row2  +--------+------------+
```

Ví dụ của đề `sumRegion(2, 1, 4, 3)`: `A = P[5][4] = 38`, `B = P[2][4] = 24`, `C = P[5][1] = 14`, `D = P[2][1] = 8` → `38 - 24 - 14 + 8 = 8` — đúng Output đầu tiên của đề.

Quy tắc nhớ chỉ số: góc "**hết** vùng" dùng `row2 + 1`, `col2 + 1`; góc "**trước** vùng" dùng `row1`, `col1` (không cộng 1) — vì `P[i][...]` là tổng của `i` hàng **đầu tiên** (hàng `0..i-1`).

In [6]:
r1, c1, r2, c2 = 2, 1, 4, 3
A_ = P2[r2 + 1][c2 + 1]
B_ = P2[r1][c2 + 1]
C_ = P2[r2 + 1][c1]
D_ = P2[r1][c1]
print(f"A = {A_}, B = {B_}, C = {C_}, D = {D_}  ->  A - B - C + D = {A_ - B_ - C_ + D_}")
print("Cộng trực tiếp từng ô:",
      sum(EXAMPLE_MATRIX[r][c] for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)))

A = 38, B = 24, C = 14, D = 8  ->  A - B - C + D = 8
Cộng trực tiếp từng ô: 8


## 5. Nền tảng 4 — Ôn #307: khi mảng **thay đổi**, prefix sum bị "vỡ"

Bài **#307 Range Sum Query - Mutable** (bản 1D của bài này): `NumArray(nums)`, `update(index, val)` (**đặt** `nums[index] = val`), `sumRange(left, right)`. Example của #307:

```text
["NumArray", "sumRange", "update", "sumRange"]
[[[1, 3, 5]], [0, 2], [1, 2], [0, 2]]
Output: [null, 9, null, 8]
```

Nếu dùng prefix sum: `nums = [1, 3, 5]` → `P = [0, 1, 4, 9]`. Sau `update(1, 2)` (đổi `3` thành `2`, tức **chênh lệch** `delta = 2 - 3 = -1`), mọi tiền tố **chứa** `nums[1]` đều phải trừ 1: `P[2]` và `P[3]` → `P = [0, 1, 3, 8]`.

Tổng quát: `update(index)` làm đổi `P[index+1], P[index+2], ..., P[n]` — tức `n - index` giá trị, tệ nhất là **cả `n`** giá trị khi `index = 0`. Hỏi `O(1)` nhưng sửa `O(n)`.

Hai bài học từ #307, sẽ dùng nguyên xi cho #308:

1. `update` là **gán**: thứ thật sự "lan truyền" vào các tổng đã lưu là **`delta = val - giá_trị_cũ`**, nên ta phải **nhớ giá trị hiện tại** của từng phần tử.
2. Muốn **sửa** nhanh thì mỗi phần tử chỉ được "nằm trong" **ít** tổng được lưu sẵn; muốn **hỏi** nhanh thì mỗi câu hỏi chỉ được ghép từ **ít** tổng được lưu sẵn. Fenwick Tree (mục 11) đáp ứng **cả hai** yêu cầu.

In [7]:
def prefix_1d(nums):
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    return P

nums = [1, 3, 5]
before = prefix_1d(nums)
delta = 2 - nums[1]               # update(1, 2): gán nums[1] = 2  ->  delta = -1
nums[1] = 2
after = prefix_1d(nums)
print("P trước:", before, "| P sau:", after, "| delta =", delta)
print("Các vị trí P bị đổi:", [k for k in range(len(before)) if before[k] != after[k]])

big = list(range(1, 11))          # mảng 10 phần tử [1, 2, ..., 10]
for idx in [9, 5, 0]:
    changed = big[:]
    changed[idx] += 100
    diff = sum(x != y for x, y in zip(prefix_1d(big), prefix_1d(changed)))
    print(f"update index {idx}: số giá trị P phải sửa = {diff}")

P trước: [0, 1, 4, 9] | P sau: [0, 1, 3, 8] | delta = -1
Các vị trí P bị đổi: [2, 3]
update index 9: số giá trị P phải sửa = 1
update index 5: số giá trị P phải sửa = 5
update index 0: số giá trị P phải sửa = 10


## 6. Bức tranh lớn: đây là bài **cân bằng chi phí**

Gọi `U` = số lần `update`, `S` = số lần `sumRegion` (`U + S = Q <= 5000`). Tổng thời gian ≈ `U × (chi phí 1 update) + S × (chi phí 1 sumRegion)`.

Đề không cho biết tỉ lệ giữa `U` và `S`, nên phải chuẩn bị cho trường hợp xấu nhất: **cả 5000 lệnh đều là loại đắt hơn**. Ước lượng rất thô: Python thuần làm được cỡ `10^7` phép tính đơn giản mỗi giây, nên ta muốn tổng số bước chỉ cỡ vài triệu.

"Bản đồ" bốn cách chính sẽ đi qua (`m = n = 200`):

| Cách | update (tệ nhất) | sumRegion (tệ nhất) | `5000 ×` loại đắt hơn |
|---|---|---|---|
| 1. Ngây thơ | `O(1)` | `O(mn)` ≈ 40 000 ô | ≈ `2 × 10^8` — quá chậm |
| 2. Prefix 2D (#304) | `O(mn)` ≈ 40 000 ô | `O(1)` (4 ô) | ≈ `2 × 10^8` — quá chậm |
| 3. Prefix theo hàng | `O(n)` ≈ 200 ô | `O(m)` ≈ 200 hàng | ≈ `10^6` — qua |
| 4. Fenwick 2D | `O(log m · log n)` ≤ 64 ô | `O(log m · log n)` ≤ 4 × 49 ô | ≈ `10^6` — qua, và mở rộng tốt nhất |

Các con số của Fenwick sẽ được giải thích (và đếm bằng code) ở mục 16.

## 7. Cách 1 — Ngây thơ: lưu ma trận, hỏi thì cộng từng ô

Cách người mới nghĩ ra đầu tiên — và cũng là "trọng tài" để kiểm tra mọi cách sau:

- `__init__`: lưu một **bản sao** của ma trận: `[row[:] for row in matrix]` (`row[:]` tạo list mới cho từng hàng, để sửa bản sao không làm hỏng ma trận của người gọi).
- `update(row, col, val)`: gán thẳng `self.mat[row][col] = val` → `O(1)`.
- `sumRegion(...)`: hai vòng `for` cộng mọi ô trong hình chữ nhật → `O((row2-row1+1) × (col2-col1+1))`, tệ nhất `O(mn)`.

In [8]:
class NumMatrixNaive:
    def __init__(self, matrix):
        self.mat = [row[:] for row in matrix]      # bản sao từng hàng

    def update(self, row, col, val):
        self.mat[row][col] = val                   # O(1): gán thẳng

    def sumRegion(self, row1, col1, row2, col2):
        total = 0
        for r in range(row1, row2 + 1):            # + 1 vì range không lấy đầu mút phải
            for c in range(col1, col2 + 1):
                total += self.mat[r][c]
        return total                               # tệ nhất O(m*n)

print(run_commands(NumMatrixNaive, EX1_OPS, EX1_ARGS), "| mong đợi:", EX1_EXPECTED)

[None, 8, None, 10] | mong đợi: [None, 8, None, 10]


### Nút thắt của cách 1

Mỗi `sumRegion` cộng lại **từ đầu** những ô đã cộng ở lần trước, dù phần lớn chúng không hề đổi. Hình chữ nhật lớn nhất có `200 × 200 = 40 000` ô; `5000` lần hỏi như vậy là `2 × 10^8` phép cộng — quá chậm với Python. Ta cần **tính sẵn** để hỏi nhanh — và #304 đã dạy cách tính sẵn.

## 8. Cách 2 — Dùng lại prefix sum 2D của #304, sửa bảng `P` khi `update`

`sumRegion` giờ chỉ là 4 lần tra bảng → `O(1)`. Còn `update(row, col, val)`?

1. Tính `delta = val - self.mat[row][col]` rồi gán `self.mat[row][col] = val` (bài học 1 của #307).
2. Ô `(row, col)` nằm trong `P[i][j]` **khi và chỉ khi** `i - 1 >= row` và `j - 1 >= col`, tức `i > row` và `j > col` (hình chữ nhật góc trên-trái `i × j` phải "trùm" tới ô đó).
3. Vậy phải cộng `delta` vào **cả khối** `P[i][j]` với `row+1 <= i <= m`, `col+1 <= j <= n` — có `(m - row) × (n - col)` ô.

Ví dụ `update(3, 2, 2)` trên ma trận 5×5: sửa `P[i][j]` với `i ∈ {4, 5}`, `j ∈ {3, 4, 5}` → `2 × 3 = 6` ô. Còn `update(0, 0, ...)` thì sửa **toàn bộ** `5 × 5 = 25` ô — với ma trận 200×200 là 40 000 ô cho **một** lần update.

In [9]:
class NumMatrixPrefix2D:
    def __init__(self, matrix):
        self.mat = [row[:] for row in matrix]
        self.P = build_prefix_2d(matrix)

    def update(self, row, col, val):
        delta = val - self.mat[row][col]            # GÁN -> chênh lệch
        self.mat[row][col] = val
        m, n = len(self.mat), len(self.mat[0])
        for i in range(row + 1, m + 1):             # mọi P[i][j] có i > row và j > col
            for j in range(col + 1, n + 1):
                self.P[i][j] += delta

    def sumRegion(self, row1, col1, row2, col2):
        P = self.P
        return P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1] + P[row1][col1]

print(run_commands(NumMatrixPrefix2D, EX1_OPS, EX1_ARGS), "| mong đợi:", EX1_EXPECTED)

obj = NumMatrixPrefix2D(EXAMPLE_MATRIX)
before = [row[:] for row in obj.P]
obj.update(3, 2, 2)
print("\nCác ô P[i][j] bị sửa khi update(3, 2, 2) (dấu *):")
print("        j = " + " ".join(str(j) for j in range(6)))
for i in range(len(before)):
    print(f"  i = {i}:   " + " ".join("*" if before[i][j] != obj.P[i][j] else "." for j in range(6)))

[None, 8, None, 10] | mong đợi: [None, 8, None, 10]

Các ô P[i][j] bị sửa khi update(3, 2, 2) (dấu *):
        j = 0 1 2 3 4 5
  i = 0:   . . . . . .
  i = 1:   . . . . . .
  i = 2:   . . . . . .
  i = 3:   . . . . . .
  i = 4:   . . . * * *
  i = 5:   . . . * * *


### Nút thắt của cách 2 — hai thái cực

| | Mỗi số được lưu sẵn "chịu trách nhiệm" cho... | update sửa | sumRegion đọc |
|---|---|---|---|
| Cách 1 | đúng **1 ô** | 1 số | tới `m·n` số |
| Cách 2 | cả **hình chữ nhật góc trên-trái** | tới `m·n` số | 4 số |

Cách 1 "lười lúc ghi, vất vả lúc đọc"; cách 2 ngược lại. Với 5000 lệnh mà loại nào cũng có thể chiếm đa số, cả hai đều có thể bị **TLE** (Time Limit Exceeded — quá thời gian). Ta cần một điểm **ở giữa**.

## 9. Cách 3 — Prefix sum theo **từng hàng**: cân bằng `O(n)` / `O(m)`

Ý tưởng: thay vì một bảng prefix 2D "dính" mọi hàng vào nhau, cho **mỗi hàng một prefix sum 1D riêng** (chính là #303 áp dụng cho từng hàng):

```text
rp[r][k] = matrix[r][0] + ... + matrix[r][k-1]     (tổng k ô đầu của hàng r; rp[r][0] = 0)
```

- `update(row, col, val)`: chỉ **hàng `row`** bị ảnh hưởng → sửa `rp[row][col+1..n]` → **`O(n)`**.
- `sumRegion(row1, col1, row2, col2)`: mỗi hàng `r` từ `row1` tới `row2` đóng góp `rp[r][col2+1] - rp[r][col1]` (công thức #303) → **`O(m)`**.

Với `m = n = 200`: mỗi lệnh tốn cỡ 200 bước, `5000` lệnh ≈ `10^6` bước — đủ nhanh cho constraints của đề. Đây là lời giải "vừa đủ" mà nhiều người dùng khi phỏng vấn trước khi được hỏi tiếp "nếu ma trận lớn hơn thì sao?".

In [10]:
class NumMatrixRowPrefix:
    def __init__(self, matrix):
        self.mat = [row[:] for row in matrix]
        self.rp = []                                # rp[r] = prefix sum 1D của hàng r (dài n+1)
        for row in matrix:
            p = [0]
            for v in row:
                p.append(p[-1] + v)
            self.rp.append(p)

    def update(self, row, col, val):
        delta = val - self.mat[row][col]
        self.mat[row][col] = val
        p = self.rp[row]
        for k in range(col + 1, len(p)):            # chỉ sửa đúng 1 hàng: O(n)
            p[k] += delta

    def sumRegion(self, row1, col1, row2, col2):
        total = 0
        for r in range(row1, row2 + 1):             # mỗi hàng O(1): tổng O(m)
            total += self.rp[r][col2 + 1] - self.rp[r][col1]
        return total

print(run_commands(NumMatrixRowPrefix, EX1_OPS, EX1_ARGS), "| mong đợi:", EX1_EXPECTED)

[None, 8, None, 10] | mong đợi: [None, 8, None, 10]


### Trace cách 3 trên Example 1

`sumRegion(2, 1, 4, 3)`: cột `1..3` → mỗi hàng lấy `rp[r][4] - rp[r][1]`.

| hàng `r` | `rp[r]` | đóng góp `rp[r][4] - rp[r][1]` |
|---:|---|---:|
| 2 | `[0, 1, 3, 3, 4, 9]` | `4 - 1 = 3` |
| 3 | `[0, 4, 5, 5, 6, 13]` | `6 - 4 = 2` |
| 4 | `[0, 1, 1, 4, 4, 9]` | `4 - 1 = 3` |
| | | **tổng = 8** |

`update(3, 2, 2)`: `delta = 2 - 0 = 2`, chỉ hàng 3 đổi: `rp[3]` từ `[0, 4, 5, 5, 6, 13]` thành `[0, 4, 5, 7, 8, 15]` (sửa `k = 3, 4, 5`). Lần hỏi sau, hàng 3 đóng góp `8 - 4 = 4` → tổng `3 + 4 + 3 = 10`.

In [11]:
def trace_row_prefix(matrix, query, upd):
    obj = NumMatrixRowPrefix(matrix)
    r1, c1, r2, c2 = query

    def show_query(title):
        print(title)
        total = 0
        for r in range(r1, r2 + 1):
            hi, lo = obj.rp[r][c2 + 1], obj.rp[r][c1]
            total += hi - lo
            print(f"   hàng {r}: rp = {obj.rp[r]}  ->  rp[{c2 + 1}] - rp[{c1}] = {hi} - {lo} = {hi - lo}")
        print(f"   tổng = {total}")

    show_query(f"sumRegion{query} TRƯỚC update:")
    row, col, val = upd
    old = obj.rp[row][:]
    obj.update(row, col, val)
    changed = [k for k in range(len(old)) if old[k] != obj.rp[row][k]]
    print(f"update{upd}: hàng {row}: {old} -> {obj.rp[row]}  (sửa k = {changed})")
    show_query(f"sumRegion{query} SAU update:")

trace_row_prefix(EXAMPLE_MATRIX, (2, 1, 4, 3), (3, 2, 2))

sumRegion(2, 1, 4, 3) TRƯỚC update:
   hàng 2: rp = [0, 1, 3, 3, 4, 9]  ->  rp[4] - rp[1] = 4 - 1 = 3
   hàng 3: rp = [0, 4, 5, 5, 6, 13]  ->  rp[4] - rp[1] = 6 - 4 = 2
   hàng 4: rp = [0, 1, 1, 4, 4, 9]  ->  rp[4] - rp[1] = 4 - 1 = 3
   tổng = 8
update(3, 2, 2): hàng 3: [0, 4, 5, 5, 6, 13] -> [0, 4, 5, 7, 8, 15]  (sửa k = [3, 4, 5])
sumRegion(2, 1, 4, 3) SAU update:
   hàng 2: rp = [0, 1, 3, 3, 4, 9]  ->  rp[4] - rp[1] = 4 - 1 = 3
   hàng 3: rp = [0, 4, 5, 7, 8, 15]  ->  rp[4] - rp[1] = 8 - 4 = 4
   hàng 4: rp = [0, 1, 1, 4, 4, 9]  ->  rp[4] - rp[1] = 4 - 1 = 3
   tổng = 10


## 10. Insight: "mỗi số lưu sẵn chịu trách nhiệm cho một đoạn" — chọn độ dài đoạn thật khéo

Cách 3 cân bằng nhưng vẫn **tuyến tính** theo một chiều: `O(n)` cho update, `O(m)` cho truy vấn. Nếu ma trận lớn hơn — hoặc ở bài 1D #307 với tới `3·10^4` phần tử — ta muốn tốt hơn nữa. Hãy tạm quay về **1 chiều** (một hàng) để tìm ý tưởng.

Với mảng 1D, mọi cách "tính sẵn tổng" đều có dạng: lưu một số con số, **mỗi con số = tổng của một đoạn** nào đó. Khi đó:

- `update(i)` phải sửa **mọi con số có đoạn chứa `i`**;
- `prefix(i)` (tổng `i` phần tử đầu) phải **ghép** từ các đoạn được lưu.

So sánh 3 cách chọn đoạn với `n = 8` (đánh số `1..8`), câu hỏi `prefix(7)` và thao tác `update(3)`:

| Cách chọn đoạn | Các đoạn được lưu | `prefix(7)` đọc mấy số | `update(3)` sửa mấy số |
|---|---|---:|---:|
| Mỗi đoạn dài 1 (như cách 1) | `[1..1], [2..2], ..., [8..8]` | 7 | 1 |
| Mọi tiền tố (prefix sum) | `[1..1], [1..2], ..., [1..8]` | 1 | 6 (`[1..3]` tới `[1..8]`) |
| **Đoạn dài lũy thừa của 2** (Fenwick) | `[1..1], [1..2], [3..3], [1..4], [5..5], [5..6], [7..7], [1..8]` | **3** (`[7..7] + [5..6] + [1..4]`) | **3** (`[3..3], [1..4], [1..8]`) |

Với `n = 10^6`: hai cách đầu có một thao tác tốn tới `10^6` bước, còn Fenwick chỉ cỡ **20** bước cho **cả hai** thao tác (`log2(10^6) ≈ 20`). Đó là toàn bộ "phép màu" của Fenwick Tree — phần tiếp theo dạy nó từ đầu, rồi ta nâng lên 2D.

## 11. Fenwick Tree 1D từ con số 0

**Fenwick Tree** (còn gọi **Binary Indexed Tree — BIT**, "cây chỉ số nhị phân") là một mảng `tree[1..n]` (đánh số **từ 1**), trong đó `tree[i]` lưu tổng của một đoạn **kết thúc tại `i`**, có độ dài là **lũy thừa của 2**. Độ dài đó được quyết định bởi biểu diễn nhị phân của `i`.

### 11.1. Nhị phân và `lowbit`

Mỗi số nguyên dương viết được dưới dạng nhị phân (tổng các lũy thừa của 2), ví dụ `12 = 8 + 4 = 1100₂`, `6 = 4 + 2 = 110₂`, `7 = 4 + 2 + 1 = 111₂`.

**`lowbit(i)`** = giá trị của **bit 1 thấp nhất** (bit 1 nằm bên phải nhất) của `i`:

- `12 = 1100₂` → bit 1 phải nhất có giá trị 4 → `lowbit(12) = 4`;
- `6 = 110₂` → `lowbit(6) = 2`;
- `7 = 111₂` → `lowbit(7) = 1`;
- `8 = 1000₂` → `lowbit(8) = 8`.

Trong Python (và C++, Java...) tính bằng đúng một phép: **`i & -i`**. Toán tử `&` là **AND từng bit**: bit kết quả bằng 1 chỉ khi bit tương ứng của **cả hai** số đều bằng 1. Còn `format(i, "05b")` in `i` dạng nhị phân, đủ 5 chữ số.

In [12]:
for i in range(1, 17):
    print(f"i = {i:>2} = {i:05b}₂    lowbit(i) = i & -i = {i & -i:>2}")

i =  1 = 00001₂    lowbit(i) = i & -i =  1
i =  2 = 00010₂    lowbit(i) = i & -i =  2
i =  3 = 00011₂    lowbit(i) = i & -i =  1
i =  4 = 00100₂    lowbit(i) = i & -i =  4
i =  5 = 00101₂    lowbit(i) = i & -i =  1
i =  6 = 00110₂    lowbit(i) = i & -i =  2
i =  7 = 00111₂    lowbit(i) = i & -i =  1
i =  8 = 01000₂    lowbit(i) = i & -i =  8
i =  9 = 01001₂    lowbit(i) = i & -i =  1
i = 10 = 01010₂    lowbit(i) = i & -i =  2
i = 11 = 01011₂    lowbit(i) = i & -i =  1
i = 12 = 01100₂    lowbit(i) = i & -i =  4
i = 13 = 01101₂    lowbit(i) = i & -i =  1
i = 14 = 01110₂    lowbit(i) = i & -i =  2
i = 15 = 01111₂    lowbit(i) = i & -i =  1
i = 16 = 10000₂    lowbit(i) = i & -i = 16


### 11.2. Vì sao `i & -i` lại ra bit thấp nhất?

Máy tính lưu số âm theo **bù 2** (two's complement): `-i = (~i) + 1`, trong đó `~i` là **lật mọi bit** của `i` (0 thành 1, 1 thành 0).

Lấy `i = 12` (xem 8 bit):

```text
 i        = 0000 1100
~i        = 1111 0011      (lật mọi bit)
~i + 1    = 1111 0100      (cộng 1: hai số 1 ở đuôi của ~i thành 0, "nhớ" lên tới số 0 đầu tiên)
i & -i    = 0000 0100  = 4
```

- Phía **dưới** bit 1 thấp nhất của `i`: `i` toàn 0, `-i` cũng toàn 0 → AND ra 0.
- **Tại** bit đó: cả hai đều là 1 → giữ lại.
- Phía **trên**: `-i` đúng là bản lật của `i` → AND ra 0.

Chỉ còn đúng một bit: `lowbit(i)`. (Số nguyên Python không giới hạn số bit, nhưng phép `&` với số âm vẫn hành xử như bù 2 "dài vô hạn", nên `i & -i` vẫn đúng.) Cell dưới in 8 bit thấp bằng `x & 0xFF` (`0xFF = 255 = 11111111₂`, AND với nó là "giữ lại 8 bit cuối").

In [13]:
def bits8(x):
    return format(x & 0xFF, "08b")     # 8 bit thấp; số âm hiện ở dạng bù 2

for i in [12, 6, 7, 8]:
    print(f"i = {i:>2}:  i = {bits8(i)}   ~i = {bits8(~i)}   -i = {bits8(-i)}   "
          f"i & -i = {bits8(i & -i)} = {i & -i}")

i = 12:  i = 00001100   ~i = 11110011   -i = 11110100   i & -i = 00000100 = 4
i =  6:  i = 00000110   ~i = 11111001   -i = 11111010   i & -i = 00000010 = 2
i =  7:  i = 00000111   ~i = 11111000   -i = 11111001   i & -i = 00000001 = 1
i =  8:  i = 00001000   ~i = 11110111   -i = 11111000   i & -i = 00001000 = 8


### 11.3. "Cây trách nhiệm": `tree[i]` phụ trách đoạn nào?

Quy tắc duy nhất cần nhớ (`a[1..n]` là mảng gốc, đánh số từ 1):

```text
tree[i] = a[i - lowbit(i) + 1] + ... + a[i]      (đoạn dài lowbit(i), KẾT THÚC tại i)
```

Bảng cho `n = 8`:

| `i` | nhị phân | `lowbit(i)` | `tree[i]` phụ trách | độ dài |
|---:|:---:|---:|---|---:|
| 1 | `0001` | 1 | `[1..1]` | 1 |
| 2 | `0010` | 2 | `[1..2]` | 2 |
| 3 | `0011` | 1 | `[3..3]` | 1 |
| 4 | `0100` | 4 | `[1..4]` | 4 |
| 5 | `0101` | 1 | `[5..5]` | 1 |
| 6 | `0110` | 2 | `[5..6]` | 2 |
| 7 | `0111` | 1 | `[7..7]` | 1 |
| 8 | `1000` | 8 | `[1..8]` | 8 |

Quan sát: số **lẻ** luôn phụ trách đúng 1 phần tử (bit cuối là 1); các `i` có cùng `lowbit = L` là `L, 3L, 5L, ...`. Vẽ các đoạn chồng lên nhau ta thấy một "cái cây": đoạn dài ôm trọn các đoạn ngắn bên trong nó — `[1..8]` ôm `[1..4]` và `[5..6]`, `[7..7]`; `[1..4]` ôm `[1..2]` và `[3..3]`...

Ta dùng mảng demo `a = [3, 1, 4, 1, 5, 9, 2, 6]` (trong Python là `A[0..7]`, ứng với `a[1..8]`). Cell sau tính `tree` **thẳng theo định nghĩa** (cộng cả đoạn — chậm, chỉ để hiểu) và vẽ các đoạn theo từng "tầng" độ dài 8, 4, 2, 1.

In [14]:
A = [3, 1, 4, 1, 5, 9, 2, 6]       # a[1..8] (1-index) = A[0..7] (0-index)
n = len(A)

def lowbit(i):
    return i & -i

tree_def = [0] * (n + 1)                   # tree_def[0] bỏ trống
for i in range(1, n + 1):
    lo = i - lowbit(i) + 1                 # đầu đoạn (1-index)
    tree_def[i] = sum(A[lo - 1:i])         # a[lo..i] (1-index) = A[lo-1 .. i-1] (0-index)

W = 4                                      # mỗi vị trí chiếm 4 ký tự khi vẽ
print(" " * 16 + "".join(f"{k:>{W}}" for k in range(1, n + 1)) + "   <- vị trí k (1-index)")
print(" " * 16 + "".join(f"{v:>{W}}" for v in A) + "   <- a[k]")
L = 8
while L >= 1:                              # vẽ từng tầng: độ dài 8, 4, 2, 1
    for i in range(L, n + 1, 2 * L):       # các i có lowbit(i) = L: L, 3L, 5L, ...
        lo = i - L + 1
        bar = " " * (W * (lo - 1)) + " [" + "=" * (W * L - 3) + "]"
        print(f"tree[{i}] = {tree_def[i]:>3}   " + bar)
    L //= 2
print("tree_def[1..8] =", tree_def[1:])

                   1   2   3   4   5   6   7   8   <- vị trí k (1-index)
                   3   1   4   1   5   9   2   6   <- a[k]
tree[8] =  31    [=============================]
tree[4] =   9    [=============]
tree[2] =   4    [=====]
tree[6] =  14                    [=====]
tree[1] =   3    [=]
tree[3] =   4            [=]
tree[5] =   5                    [=]
tree[7] =   2                            [=]
tree_def[1..8] = [3, 4, 4, 9, 5, 14, 2, 31]


### 11.4. Truy vấn tiền tố `prefix(i)`: nhảy `i -= lowbit(i)`

`prefix(i) = a[1] + ... + a[i]`. Bắt đầu từ `tree[i]` (phủ đoạn `[i - lowbit(i) + 1 .. i]`), phần còn thiếu là `[1 .. i - lowbit(i)]` — lại là **một tiền tố**! Vậy lặp lại:

```python
total = 0
while i > 0:
    total += tree[i]
    i -= lowbit(i)      # nhảy tới ngay trước đoạn vừa cộng
```

Ví dụ `prefix(7)` với mảng demo:

| Bước | `i` (nhị phân) | cộng | đoạn | `i` mới |
|---:|---|---:|---|---|
| 1 | 7 (`111`) | `tree[7] = 2` | `[7..7]` | 7 − 1 = 6 |
| 2 | 6 (`110`) | `tree[6] = 14` | `[5..6]` | 6 − 2 = 4 |
| 3 | 4 (`100`) | `tree[4] = 9` | `[1..4]` | 4 − 4 = 0 → dừng |

Tổng `2 + 14 + 9 = 25 = 3+1+4+1+5+9+2`.

Để ý: `7 = 111₂` có **ba bit 1** → đúng 3 bước. Mỗi bước **xóa bit 1 thấp nhất** (`111 → 110 → 100 → 000`), và độ dài các đoạn chính là giá trị các bit đó: `1 + 2 + 4 = 7`. Các đoạn **nối liền, không chồng nhau**, phủ đúng `[1..7]`. Số bước = số bit 1 của `i`, luôn `<= log2(n) + 1`.

In [15]:
tree = tree_def[:]                 # dùng mảng tree vừa tính theo định nghĩa

def trace_prefix(tree, i):
    print(f"prefix({i}):")
    total = 0
    while i > 0:
        lo = i - lowbit(i) + 1
        total += tree[i]
        print(f"   i = {i} ({i:04b}₂): cộng tree[{i}] = {tree[i]:>2} (đoạn [{lo}..{i}]) "
              f"-> tổng = {total:>2};  i -= {lowbit(i)} -> {i - lowbit(i)}")
        i -= lowbit(i)
    return total

for q in [7, 8, 6]:
    got = trace_prefix(tree, q)
    print(f"   kiểm tra: sum(A[:{q}]) = {sum(A[:q])}  ->  {'khớp' if got == sum(A[:q]) else 'SAI'}")

prefix(7):
   i = 7 (0111₂): cộng tree[7] =  2 (đoạn [7..7]) -> tổng =  2;  i -= 1 -> 6
   i = 6 (0110₂): cộng tree[6] = 14 (đoạn [5..6]) -> tổng = 16;  i -= 2 -> 4
   i = 4 (0100₂): cộng tree[4] =  9 (đoạn [1..4]) -> tổng = 25;  i -= 4 -> 0
   kiểm tra: sum(A[:7]) = 25  ->  khớp
prefix(8):
   i = 8 (1000₂): cộng tree[8] = 31 (đoạn [1..8]) -> tổng = 31;  i -= 8 -> 0
   kiểm tra: sum(A[:8]) = 31  ->  khớp
prefix(6):
   i = 6 (0110₂): cộng tree[6] = 14 (đoạn [5..6]) -> tổng = 14;  i -= 2 -> 4
   i = 4 (0100₂): cộng tree[4] =  9 (đoạn [1..4]) -> tổng = 23;  i -= 4 -> 0
   kiểm tra: sum(A[:6]) = 23  ->  khớp


### 11.5. Cập nhật `add(i, delta)`: nhảy `i += lowbit(i)`

Khi `a[i]` tăng thêm `delta`, phải sửa **mọi** `tree[j]` có đoạn chứa `i`. Nhìn bảng 11.3: vị trí 3 nằm trong `[3..3]`, `[1..4]`, `[1..8]` → cần sửa `tree[3], tree[4], tree[8]`. Và thật kỳ diệu: `3 → 3 + 1 = 4 → 4 + 4 = 8 → 8 + 8 = 16 > n` (dừng).

```python
while i <= n:
    tree[i] += delta
    i += lowbit(i)      # nhảy lên đoạn "cha" ngay trên, đoạn này ôm trọn đoạn cũ
```

Vì sao đúng (trực giác):

- Đặt `j = i + lowbit(i)`. Phép cộng làm bit thấp nhất của `i` "nhớ" lên trên, nên `lowbit(j) >= 2 · lowbit(i)`. Do đó đoạn của `j` bắt đầu ở `j - lowbit(j) + 1 <= i - lowbit(i) + 1`: đoạn của `j` **ôm trọn** đoạn của `i`, nên vẫn chứa vị trí ban đầu.
- Các số nằm **giữa** `i` và `i + lowbit(i)` có `lowbit` nhỏ hơn, đoạn của chúng bắt đầu **sau** `i` → không chứa vị trí ban đầu, bỏ qua là đúng.
- Mỗi bước `lowbit` ít nhất gấp đôi → tối đa `log2(n) + 1` bước.

Cell sau kiểm tra bằng **vét cạn**: với mọi vị trí `p` từ 1 tới 8, so tập các `tree[j]` có đoạn chứa `p` (tìm thẳng theo định nghĩa) với đường nhảy `p, p + lowbit(p), ...`; phần sau của cùng cell trace `add(3, +5)` (tức `a[3]` từ 4 thành 9) và so cây sau khi sửa với cây tính lại từ đầu.

In [16]:
def update_path(p, n):
    path = []
    while p <= n:
        path.append(p)
        p += lowbit(p)
    return path

for p in range(1, n + 1):
    covering = [j for j in range(1, n + 1) if j - lowbit(j) + 1 <= p <= j]   # vét cạn
    print(f"p = {p}: đường nhảy {str(update_path(p, n)):<14} các tree[j] chứa p (vét cạn) {covering}")
    assert update_path(p, n) == covering

def trace_add(tree, i, delta):
    n = len(tree) - 1
    print(f"add({i}, {delta:+d}):")
    while i <= n:
        lo = i - lowbit(i) + 1
        print(f"   tree[{i}] (đoạn [{lo}..{i}]): {tree[i]:>2} -> {tree[i] + delta:>2};  "
              f"i += {lowbit(i)} -> {i + lowbit(i)}")
        tree[i] += delta
        i += lowbit(i)
    print(f"   dừng vì i > n = {n}")

tree = tree_def[:]
trace_add(tree, 3, +5)                     # a[3]: 4 -> 9
A2 = A[:]
A2[2] += 5
expect = [0] + [sum(A2[k - lowbit(k):k]) for k in range(1, n + 1)]
print("tree sau add    :", tree[1:])
print("tính lại từ đầu :", expect[1:], "->", "khớp" if tree == expect else "SAI")

p = 1: đường nhảy [1, 2, 4, 8]   các tree[j] chứa p (vét cạn) [1, 2, 4, 8]
p = 2: đường nhảy [2, 4, 8]      các tree[j] chứa p (vét cạn) [2, 4, 8]
p = 3: đường nhảy [3, 4, 8]      các tree[j] chứa p (vét cạn) [3, 4, 8]
p = 4: đường nhảy [4, 8]         các tree[j] chứa p (vét cạn) [4, 8]
p = 5: đường nhảy [5, 6, 8]      các tree[j] chứa p (vét cạn) [5, 6, 8]
p = 6: đường nhảy [6, 8]         các tree[j] chứa p (vét cạn) [6, 8]
p = 7: đường nhảy [7, 8]         các tree[j] chứa p (vét cạn) [7, 8]
p = 8: đường nhảy [8]            các tree[j] chứa p (vét cạn) [8]
add(3, +5):
   tree[3] (đoạn [3..3]):  4 ->  9;  i += 1 -> 4
   tree[4] (đoạn [1..4]):  9 -> 14;  i += 4 -> 8
   tree[8] (đoạn [1..8]): 31 -> 36;  i += 8 -> 16
   dừng vì i > n = 8
tree sau add    : [3, 4, 9, 14, 5, 14, 2, 36]
tính lại từ đầu : [3, 4, 9, 14, 5, 14, 2, 36] -> khớp


### 11.6. Xây cây ban đầu: hai cách

- **Cách đơn giản** `O(n log n)`: bắt đầu `tree` toàn 0, rồi `add(k, a[k])` cho từng `k` — coi như "thêm lần lượt từng phần tử".
- **Cách tuyến tính** `O(n)`: chép `tree[i] = a[i]`, rồi duyệt `i = 1..n` **tăng dần**, đẩy tổng của `tree[i]` lên **cha trực tiếp** `p = i + lowbit(i)` (nếu `p <= n`): `tree[p] += tree[i]`.

Vì sao cách tuyến tính đúng? Các "con" của `p` (những `i` có `i + lowbit(i) = p`) đều **nhỏ hơn** `p`, nên khi vòng lặp tới `p`, mọi con đã đẩy xong vào `tree[p]` — `tree[p]` đã đủ tổng đoạn của nó trước khi chính nó được đẩy tiếp lên trên. Ví dụ `tree[8] = a[8] + tree[4] + tree[6] + tree[7]` = `6 + 9 + 14 + 2 = 31`.

Hai cách (và định nghĩa) phải cho **cùng một mảng** — cell sau kiểm tra trên mảng demo và 200 mảng ngẫu nhiên. `enumerate(nums, start=1)` cho cặp `(k, nums[k-1])` với `k` đếm từ 1.

In [17]:
import random

def build_by_add(nums):
    n = len(nums)
    tree = [0] * (n + 1)
    for k, v in enumerate(nums, start=1):      # k = 1..n
        i = k
        while i <= n:
            tree[i] += v
            i += lowbit(i)
    return tree

def build_linear(nums):
    n = len(nums)
    tree = [0] + nums[:]                       # tree[i] = a[i]
    for i in range(1, n + 1):
        p = i + lowbit(i)                      # cha trực tiếp
        if p <= n:
            tree[p] += tree[i]
    return tree

print("build bằng add  :", build_by_add(A)[1:])
print("build tuyến tính:", build_linear(A)[1:])
print("theo định nghĩa :", tree_def[1:])

random.seed(11)
for _ in range(200):
    arr = [random.randint(-50, 50) for _ in range(random.randint(1, 40))]
    assert build_by_add(arr) == build_linear(arr)
print("200 mảng ngẫu nhiên: hai cách build cho cùng một cây.")

build bằng add  : [3, 4, 4, 9, 5, 14, 2, 31]
build tuyến tính: [3, 4, 4, 9, 5, 14, 2, 31]
theo định nghĩa : [3, 4, 4, 9, 5, 14, 2, 31]
200 mảng ngẫu nhiên: hai cách build cho cùng một cây.


### 11.7. Đóng gói thành class `Fenwick` (bên trong 1-index, bên ngoài 0-index)

LeetCode dùng chỉ số **từ 0**, còn Fenwick cần **từ 1** — vì `lowbit(0) = 0 & -0 = 0`, vòng `i += lowbit(i)` sẽ **đứng yên mãi mãi** (Lỗi 3). Quy ước an toàn: **chỉ đổi chỉ số ở một chỗ** — phần tử `nums[k]` (0-index) nằm ở vị trí `k + 1` trong cây.

- `add(i, delta)`: `i` là vị trí **1-index**.
- `prefix(i)`: tổng `i` phần tử đầu (`i` từ 0 tới `n`; `prefix(0) = 0` vì vòng `while` không chạy).
- `range_sum(left, right)`: tổng `nums[left..right]` (0-index, gồm hai đầu) = `prefix(right + 1) - prefix(left)` — **y hệt công thức #303**, chỉ khác `prefix` giờ tính bằng cây.

Cell dưới kiểm tra class với 300 dãy thao tác ngẫu nhiên, so với một list thường.

In [18]:
class Fenwick:
    # Fenwick Tree 1D. Bên trong dùng chỉ số 1..n; range_sum nhận chỉ số 0-index như LeetCode.
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] + list(nums)               # tree[i] = a[i] (1-index)
        for i in range(1, self.n + 1):             # build O(n): đẩy lên cha
            p = i + (i & -i)
            if p <= self.n:
                self.tree[p] += self.tree[i]

    def add(self, i, delta):                       # i: 1-index
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i

    def prefix(self, i):                           # tổng a[1..i]
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & -i
        return total

    def range_sum(self, left, right):              # tổng nums[left..right], 0-index
        return self.prefix(right + 1) - self.prefix(left)

random.seed(307)
for _ in range(300):
    arr = [random.randint(-100, 100) for _ in range(random.randint(1, 30))]
    fw = Fenwick(arr)
    for _ in range(20):
        k = random.randrange(len(arr))
        d = random.randint(-100, 100)
        arr[k] += d
        fw.add(k + 1, d)                           # 0-index k -> 1-index k+1
        l = random.randrange(len(arr))
        r = random.randrange(l, len(arr))
        assert fw.range_sum(l, r) == sum(arr[l:r + 1])
print("Fenwick 1D khớp với list thường trên 300 x 20 thao tác ngẫu nhiên.")

Fenwick 1D khớp với list thường trên 300 x 20 thao tác ngẫu nhiên.


### 11.8. Giải #307 bằng Fenwick — và cái bẫy "gán chứ không cộng"

`update(index, val)` của #307 (và của #308) **gán** giá trị mới, trong khi Fenwick chỉ biết **cộng thêm** (`add`). Cầu nối:

```python
delta = val - self.nums[index]   # chênh lệch giữa giá trị MỚI và giá trị HIỆN TẠI
self.nums[index] = val           # nhớ giá trị mới cho lần update sau
self.bit.add(index + 1, delta)   # đổi sang 1-index
```

Vì vậy phải giữ mảng `self.nums` (giá trị hiện tại của từng phần tử). Fenwick không lưu trực tiếp `a[i]` — muốn đọc phải tính `range_sum(i, i)`, tốn `O(log n)`; giữ một mảng riêng vừa đơn giản vừa nhanh.

Test thứ hai dưới đây gán **hai lần cùng một giá trị** vào cùng một chỗ: lần hai phải có `delta = 0`.

In [19]:
class NumArrayFenwick:                       # lời giải #307
    def __init__(self, nums):
        self.nums = nums[:]
        self.bit = Fenwick(nums)

    def update(self, index, val):
        delta = val - self.nums[index]       # GÁN -> chuyển thành CỘNG delta
        self.nums[index] = val
        self.bit.add(index + 1, delta)       # 0-index -> 1-index

    def sumRange(self, left, right):
        return self.bit.range_sum(left, right)

print(run_commands(NumArrayFenwick, ["NumArray", "sumRange", "update", "sumRange"],
                   [[[1, 3, 5]], [0, 2], [1, 2], [0, 2]]))       # đề #307: [null, 9, null, 8]
print(run_commands(NumArrayFenwick, ["NumArray", "update", "update", "sumRange"],
                   [[[1, 3, 5]], [1, 10], [1, 10], [0, 2]]))     # 1 + 10 + 5 = 16

[None, 9, None, 8]
[None, None, None, 16]


## 12. Fenwick Tree 2D: "Fenwick của Fenwick"

### 12.1. Mỗi ô `tree[i][j]` phụ trách một **khối chữ nhật**

Áp quy tắc 1D cho **cả hai chiều** (mọi chỉ số ở đây là 1-index):

```text
tree[i][j] = tổng các ô (hàng r, cột c) với
             i - lowbit(i) + 1 <= r <= i      (đoạn hàng của i)
             j - lowbit(j) + 1 <= c <= j      (đoạn cột của j)
```

Ví dụ trên ma trận 5×5 của đề:

- `tree[4][4]` phụ trách hàng `1..4` × cột `1..4` (khối 4×4);
- `tree[4][3]` phụ trách hàng `1..4` × cột `3..3` (một cột dài 4 ô);
- `tree[5][4]` phụ trách hàng `5..5` × cột `1..4` (một hàng dài 4 ô);
- `tree[3][5]` phụ trách đúng một ô `(3, 5)`.

### 12.2. Hai thao tác = hai vòng nhảy lồng nhau

```python
def add(i, j, delta):              # ô (i, j) 1-index tăng thêm delta
    while i <= m:
        jj = j
        while jj <= n:
            tree[i][jj] += delta
            jj += jj & -jj
        i += i & -i

def prefix(i, j):                  # tổng khối hàng 1..i × cột 1..j
    total = 0
    while i > 0:
        jj = j
        while jj > 0:
            total += tree[i][jj]
            jj -= jj & -jj
        i -= i & -i
    return total
```

Vì sao? Khối `tree[i][j]` chứa ô `(r, c)` **khi và chỉ khi** đoạn hàng của `i` chứa `r` **và** đoạn cột của `j` chứa `c`. Theo mục 11.5, các `i` có đoạn chứa `r` chính là đường nhảy `r, r + lowbit(r), ...`; các `j` tương tự với `c`. Vậy các khối cần sửa = **mọi cặp** (tích) của hai đường nhảy → hai vòng lồng nhau. Lập luận y hệt cho `prefix`: khối `1..i × 1..j` được chia thành (các đoạn hàng của `i`) × (các đoạn cột của `j`), rời nhau và phủ kín.

Lưu ý: vòng trong phải dùng **biến mới** `jj`, **khởi tạo lại** từ `j` cho mỗi hàng — nếu dùng chung `j`, hàng thứ hai sẽ bắt đầu từ chỗ hàng thứ nhất đã dừng (sai, và là lỗi rất hay gặp).

Cell dưới in khối phụ trách của **mọi** ô `tree[i][j]` trên ma trận 5×5 và tô hai khối `tree[4][4]`, `tree[4][3]`.

In [20]:
def block_of(i, j):
    # Khối (1-index) mà tree[i][j] phụ trách: (hàng từ, hàng tới, cột từ, cột tới)
    return (i - lowbit(i) + 1, i, j - lowbit(j) + 1, j)

M5 = N5 = 5
print("Khối mà tree[i][j] phụ trách (h = hàng, c = cột, 1-index):")
for i in range(1, M5 + 1):
    cells = []
    for j in range(1, N5 + 1):
        a, b, c, d = block_of(i, j)
        cells.append(f"h{a}-{b},c{c}-{d}")
    print(f"  i={i}: " + "  ".join(f"{x:<10}" for x in cells))

def draw_block(i, j, m, n):
    a, b, c, d = block_of(i, j)
    print(f"\ntree[{i}][{j}] phụ trách hàng {a}..{b} × cột {c}..{d} (dấu #):")
    for r in range(1, m + 1):
        print("   " + " ".join("#" if a <= r <= b and c <= cc <= d else "." for cc in range(1, n + 1)))

draw_block(4, 4, M5, N5)
draw_block(4, 3, M5, N5)

Khối mà tree[i][j] phụ trách (h = hàng, c = cột, 1-index):
  i=1: h1-1,c1-1   h1-1,c1-2   h1-1,c3-3   h1-1,c1-4   h1-1,c5-5 
  i=2: h1-2,c1-1   h1-2,c1-2   h1-2,c3-3   h1-2,c1-4   h1-2,c5-5 
  i=3: h3-3,c1-1   h3-3,c1-2   h3-3,c3-3   h3-3,c1-4   h3-3,c5-5 
  i=4: h1-4,c1-1   h1-4,c1-2   h1-4,c3-3   h1-4,c1-4   h1-4,c5-5 
  i=5: h5-5,c1-1   h5-5,c1-2   h5-5,c3-3   h5-5,c1-4   h5-5,c5-5 

tree[4][4] phụ trách hàng 1..4 × cột 1..4 (dấu #):
   # # # # .
   # # # # .
   # # # # .
   # # # # .
   . . . . .

tree[4][3] phụ trách hàng 1..4 × cột 3..3 (dấu #):
   . . # . .
   . . # . .
   . . # . .
   . . # . .
   . . . . .


### 12.3. Từ `prefix` tới `sumRegion`: bao hàm – loại trừ như #304

`prefix(i, j)` của Fenwick đóng đúng vai trò `P[i][j]` của #304 (cả hai đều là tổng khối "`i` hàng đầu × `j` cột đầu"). Vì vậy công thức truy vấn **giữ nguyên**:

```text
sumRegion(row1, col1, row2, col2) = prefix(row2+1, col2+1) - prefix(row1, col2+1)
                                  - prefix(row2+1, col1)   + prefix(row1, col1)
```

Khác biệt duy nhất: `P[i][j]` đọc trong `O(1)` nhưng sửa trong `O(mn)`; `prefix(i, j)` đọc trong `O(log m · log n)` và sửa cũng `O(log m · log n)`.

### 12.4. Thiết kế class (bản demo, dễ đọc)

- `self.nums`: bản sao ma trận — để tính `delta` (bài học 1 của #307).
- `self.tree`: mảng `(m+1) × (n+1)` toàn 0 (hàng 0, cột 0 không dùng).
- `__init__`: gọi `_add(r+1, c+1, matrix[r][c])` cho mọi ô (cách build đơn giản; bản nộp ở mục 15 dùng cách tuyến tính).
- `update`: `delta = val - self.nums[row][col]`, gán `self.nums[row][col] = val`, rồi `_add(row+1, col+1, delta)`.
- `sumRegion`: 4 lần `_prefix`.

Tên bắt đầu bằng `_` (như `_add`) là quy ước Python cho "hàm nội bộ", người dùng class không cần gọi.

In [21]:
class NumMatrixFenwick2D:
    def __init__(self, matrix):
        self.m, self.n = len(matrix), len(matrix[0])
        self.nums = [row[:] for row in matrix]
        self.tree = [[0] * (self.n + 1) for _ in range(self.m + 1)]
        for r in range(self.m):
            for c in range(self.n):
                self._add(r + 1, c + 1, matrix[r][c])    # build đơn giản

    def _add(self, i, j, delta):          # (i, j) là 1-index
        while i <= self.m:
            jj = j                        # vòng trong bắt đầu lại từ j cho mỗi hàng
            while jj <= self.n:
                self.tree[i][jj] += delta
                jj += jj & -jj
            i += i & -i

    def _prefix(self, i, j):              # tổng khối hàng 1..i × cột 1..j
        total = 0
        while i > 0:
            jj = j
            while jj > 0:
                total += self.tree[i][jj]
                jj -= jj & -jj
            i -= i & -i
        return total

    def update(self, row, col, val):
        delta = val - self.nums[row][col]     # GÁN -> delta
        self.nums[row][col] = val
        self._add(row + 1, col + 1, delta)    # 0-index -> 1-index

    def sumRegion(self, row1, col1, row2, col2):
        return (self._prefix(row2 + 1, col2 + 1) - self._prefix(row1, col2 + 1)
                - self._prefix(row2 + 1, col1) + self._prefix(row1, col1))

print(run_commands(NumMatrixFenwick2D, EX1_OPS, EX1_ARGS), "| mong đợi:", EX1_EXPECTED)

[None, 8, None, 10] | mong đợi: [None, 8, None, 10]


## 13. Trace cách chính trên Example 1

### 13.1. `update(3, 2, 2)` chạm vào những ô `tree` nào?

Ô 0-index `(3, 2)` → 1-index `(4, 3)`. Giá trị cũ `0`, mới `2` → `delta = 2`.

- Đường nhảy hàng: `4 → 4 + 4 = 8 > 5` (dừng) → chỉ hàng `{4}`.
- Đường nhảy cột: `3 → 3 + 1 = 4 → 4 + 4 = 8 > 5` → cột `{3, 4}`.

| Bước | `i` | `j` | `tree[i][j]` phụ trách | Trước | Sau (`+2`) |
|---:|---:|---:|---|---:|---:|
| 1 | 4 | 3 | hàng 1..4 × cột 3..3 | 4 | 6 |
| 2 | 4 | 4 | hàng 1..4 × cột 1..4 | 34 | 36 |

Chỉ 2 ô! Lý do: ma trận chỉ có 5 hàng, nên khối hàng chứa hàng 4 chỉ có `[1..4]` (khối `[1..8]` không tồn tại). Để thấy đường nhảy "dài", cell sau trace thêm `update(0, 0, 10)` — ô góc, `delta = 10 - 3 = 7`: hàng `1 → 2 → 4`, cột `1 → 2 → 4` → `3 × 3 = 9` ô.

In [22]:
def trace_fenwick2d_update(obj, row, col, val):
    delta = val - obj.nums[row][col]
    print(f"update({row}, {col}, {val}): cũ = {obj.nums[row][col]}, delta = {delta}, "
          f"ô 1-index = ({row + 1}, {col + 1})")
    i, step = row + 1, 0
    while i <= obj.m:
        j = col + 1
        while j <= obj.n:
            step += 1
            a, b, c, d = block_of(i, j)
            print(f"   bước {step}: tree[{i}][{j}] (hàng {a}..{b} × cột {c}..{d}): "
                  f"{obj.tree[i][j]:>2} -> {obj.tree[i][j] + delta:>2}")
            j += lowbit(j)
        i += lowbit(i)
    obj.update(row, col, val)                 # cập nhật thật (đúng các bước vừa in)

obj = NumMatrixFenwick2D(EXAMPLE_MATRIX)
trace_fenwick2d_update(obj, 3, 2, 2)
print()
trace_fenwick2d_update(obj, 0, 0, 10)

update(3, 2, 2): cũ = 0, delta = 2, ô 1-index = (4, 3)
   bước 1: tree[4][3] (hàng 1..4 × cột 3..3):  4 ->  6
   bước 2: tree[4][4] (hàng 1..4 × cột 1..4): 34 -> 36

update(0, 0, 10): cũ = 3, delta = 7, ô 1-index = (1, 1)
   bước 1: tree[1][1] (hàng 1..1 × cột 1..1):  3 -> 10
   bước 2: tree[1][2] (hàng 1..1 × cột 1..2):  3 -> 10
   bước 3: tree[1][4] (hàng 1..1 × cột 1..4):  8 -> 15
   bước 4: tree[2][1] (hàng 1..2 × cột 1..1):  8 -> 15
   bước 5: tree[2][2] (hàng 1..2 × cột 1..2): 14 -> 21
   bước 6: tree[2][4] (hàng 1..2 × cột 1..4): 24 -> 31
   bước 7: tree[4][1] (hàng 1..4 × cột 1..1): 13 -> 20
   bước 8: tree[4][2] (hàng 1..4 × cột 1..2): 22 -> 29
   bước 9: tree[4][4] (hàng 1..4 × cột 1..4): 36 -> 43


### 13.2. `sumRegion(2, 1, 4, 3)` sau `update(3, 2, 2)`: 4 lần `prefix`

(Trace này dùng một đối tượng mới chỉ chạy đúng các lệnh của Example 1, không có `update(0, 0, 10)` ở trên.)

Đổi sang 1-index: `prefix(5, 4) − prefix(2, 4) − prefix(5, 1) + prefix(2, 1)`.

| Góc | Lời gọi | Đường nhảy hàng × cột | Các ô `tree` được cộng | Giá trị |
|---|---|---|---|---:|
| A (+) | `prefix(5, 4)` | `{5, 4} × {4}` | `tree[5][4] = 4`, `tree[4][4] = 36` | 40 |
| B (−) | `prefix(2, 4)` | `{2} × {4}` | `tree[2][4] = 24` | 24 |
| C (−) | `prefix(5, 1)` | `{5, 4} × {1}` | `tree[5][1] = 1`, `tree[4][1] = 13` | 14 |
| D (+) | `prefix(2, 1)` | `{2} × {1}` | `tree[2][1] = 8` | 8 |

Kết quả: `40 − 24 − 14 + 8 = 10` — khớp Output của đề. So với **trước** update (`38 − 24 − 14 + 8 = 8`), chỉ góc A đổi (38 → 40), vì chỉ khối A chứa ô vừa sửa. Mỗi `prefix` chỉ cộng 1–2 ô `tree` thay vì hàng chục ô của ma trận.

In [23]:
def trace_prefix_2d(obj, i, j):
    total, parts = 0, []
    while i > 0:
        jj = j
        while jj > 0:
            total += obj.tree[i][jj]
            parts.append(f"tree[{i}][{jj}]={obj.tree[i][jj]}")
            jj -= lowbit(jj)
        i -= lowbit(i)
    return total, parts

def trace_fenwick2d_sumregion(obj, row1, col1, row2, col2):
    corners = [("A", +1, row2 + 1, col2 + 1), ("B", -1, row1, col2 + 1),
               ("C", -1, row2 + 1, col1), ("D", +1, row1, col1)]
    result = 0
    for name, sign, i, j in corners:
        val, parts = trace_prefix_2d(obj, i, j)
        result += sign * val
        print(f"   {name} ({'+' if sign > 0 else '-'}) prefix({i}, {j}) = "
              f"{' + '.join(parts) if parts else '0'} = {val}")
    print(f"   => sumRegion({row1}, {col1}, {row2}, {col2}) = {result}")
    return result

obj = NumMatrixFenwick2D(EXAMPLE_MATRIX)
print("TRƯỚC update:")
trace_fenwick2d_sumregion(obj, 2, 1, 4, 3)
obj.update(3, 2, 2)
print("SAU update(3, 2, 2):")
assert trace_fenwick2d_sumregion(obj, 2, 1, 4, 3) == obj.sumRegion(2, 1, 4, 3) == 10

TRƯỚC update:
   A (+) prefix(5, 4) = tree[5][4]=4 + tree[4][4]=34 = 38
   B (-) prefix(2, 4) = tree[2][4]=24 = 24
   C (-) prefix(5, 1) = tree[5][1]=1 + tree[4][1]=13 = 14
   D (+) prefix(2, 1) = tree[2][1]=8 = 8
   => sumRegion(2, 1, 4, 3) = 8
SAU update(3, 2, 2):
   A (+) prefix(5, 4) = tree[5][4]=4 + tree[4][4]=36 = 40
   B (-) prefix(2, 4) = tree[2][4]=24 = 24
   C (-) prefix(5, 1) = tree[5][1]=1 + tree[4][1]=13 = 14
   D (+) prefix(2, 1) = tree[2][1]=8 = 8
   => sumRegion(2, 1, 4, 3) = 10


## 14. Cách 5 (mở rộng) — hai lựa chọn khác

### 14.1. Mỗi hàng một Fenwick 1D: update `O(log n)`, sumRegion `O(m log n)`

Đây là "cách 3 nâng cấp": cách 3 dùng prefix sum cho mỗi hàng (update `O(n)`); giờ thay bằng **Fenwick 1D cho mỗi hàng** (update `O(log n)`). Truy vấn vẫn đi qua từng hàng `row1..row2`, mỗi hàng `range_sum` tốn `O(log n)` → `O(m log n)`.

- Ưu điểm: dùng lại nguyên class `Fenwick` của mục 11, không cần vòng lặp lồng nhau trên cây.
- Nhược điểm: truy vấn vẫn **tuyến tính theo số hàng**, và hằng số còn lớn hơn cách 3 (mỗi hàng 2 lần `prefix` thay vì 2 lần tra bảng) — cách này lệch về phía "sửa rẻ, hỏi đắt".
- Mẹo nhỏ: nếu `m > n` thì nên dựng Fenwick theo **cột** để vòng ngoài đi qua chiều ngắn hơn.

In [24]:
class NumMatrixRowFenwick:
    def __init__(self, matrix):
        self.nums = [row[:] for row in matrix]
        self.rows = [Fenwick(row) for row in matrix]        # mỗi hàng một Fenwick 1D

    def update(self, row, col, val):
        delta = val - self.nums[row][col]
        self.nums[row][col] = val
        self.rows[row].add(col + 1, delta)                  # O(log n)

    def sumRegion(self, row1, col1, row2, col2):
        total = 0
        for r in range(row1, row2 + 1):                     # O(m log n)
            total += self.rows[r].range_sum(col1, col2)
        return total

print(run_commands(NumMatrixRowFenwick, EX1_OPS, EX1_ARGS), "| mong đợi:", EX1_EXPECTED)

[None, 8, None, 10] | mong đợi: [None, 8, None, 10]


### 14.2. Ý tưởng Segment Tree 2D (chỉ nêu ý tưởng)

**Segment tree** (cây phân đoạn) 1D là một cây nhị phân, mỗi node phụ trách một đoạn `[l..r]` và lưu tổng đoạn đó; gốc phụ trách cả mảng, hai con **chia đôi** đoạn của cha:

```text
                    [0..7]
             /                \
         [0..3]              [4..7]
        /      \            /      \
     [0..1]  [2..3]     [4..5]   [6..7]
     /  \     /  \       /  \     /  \
    0    1   2    3     4    5   6    7
```

- **update** một vị trí: sửa lá và mọi node tổ tiên của nó (một đường từ lá lên gốc) → `O(log n)`.
- **query** đoạn `[l..r]`: tách đoạn thành `O(log n)` node "nằm trọn bên trong" rồi cộng lại.

**Segment tree 2D** = "cây của cây": dựng segment tree theo **hàng**; mỗi node hàng (phụ trách dải hàng `[r1..r2]`) chứa **một segment tree theo cột** của dải đó. Update/query đều `O(log m · log n)` như Fenwick 2D, nhưng tốn bộ nhớ hơn (cỡ `4m × 4n` node nếu cài bằng mảng) và code dài hơn nhiều.

Khi nào segment tree "đáng" hơn Fenwick? Khi phép gộp **không có phép ngược**. Fenwick trả lời đoạn `[l..r]` bằng `prefix(r) − prefix(l−1)`, tức **cần phép trừ**: với tổng (và XOR) thì được; với **min/max** thì không "trừ" được → phải dùng segment tree. Bài này là tổng, nên Fenwick 2D gọn nhất. (Một hướng khác là quad-tree — chia ma trận làm 4 phần đệ quy — dễ hình dung nhưng không đảm bảo truy vấn `O(log m · log n)` trong trường hợp xấu.)

## 15. Code nộp LeetCode — cách chính (Fenwick Tree 2D)

So với bản demo `NumMatrixFenwick2D`, bản nộp có hai cải tiến nhỏ:

1. **Build tuyến tính `O(mn)`** thay vì gọi `_add` cho từng ô (`O(mn · log m · log n)`): làm đúng mẹo "đẩy lên cha" của mục 11.6, nhưng **hai lượt** — lượt 1 đẩy **theo cột** trong từng hàng (biến mỗi hàng thành một Fenwick 1D: `tree[i][j]` = tổng hàng `i` trên đoạn cột của `j`), lượt 2 đẩy **theo hàng** trong từng cột (gộp các hàng thành khối). Sau hai lượt, `tree[i][j]` đúng bằng tổng khối của nó.
2. `update` thoát sớm khi `delta == 0` (gán lại đúng giá trị cũ thì không có gì để sửa).

`from typing import List` chỉ cần khi chạy ngoài LeetCode (trên LeetCode đã có sẵn).

In [25]:
from typing import List

class NumMatrix:

    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.m, self.n = m, n
        self.nums = [row[:] for row in matrix]
        tree = [[0] * (n + 1) for _ in range(m + 1)]        # 1-index, hàng/cột 0 bỏ trống
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                tree[i][j] = matrix[i - 1][j - 1]
        for i in range(1, m + 1):                            # lượt 1: đẩy theo cột
            for j in range(1, n + 1):
                p = j + (j & -j)
                if p <= n:
                    tree[i][p] += tree[i][j]
        for j in range(1, n + 1):                            # lượt 2: đẩy theo hàng
            for i in range(1, m + 1):
                p = i + (i & -i)
                if p <= m:
                    tree[p][j] += tree[i][j]
        self.tree = tree

    def _add(self, i: int, j: int, delta: int) -> None:
        while i <= self.m:
            jj = j
            while jj <= self.n:
                self.tree[i][jj] += delta
                jj += jj & -jj
            i += i & -i

    def _prefix(self, i: int, j: int) -> int:
        total = 0
        while i > 0:
            jj = j
            while jj > 0:
                total += self.tree[i][jj]
                jj -= jj & -jj
            i -= i & -i
        return total

    def update(self, row: int, col: int, val: int) -> None:
        delta = val - self.nums[row][col]
        if delta == 0:
            return
        self.nums[row][col] = val
        self._add(row + 1, col + 1, delta)

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        return (self._prefix(row2 + 1, col2 + 1) - self._prefix(row1, col2 + 1)
                - self._prefix(row2 + 1, col1) + self._prefix(row1, col1))


# Your NumMatrix object will be instantiated and called as such:
# obj = NumMatrix(matrix)
# obj.update(row,col,val)
# param_2 = obj.sumRegion(row1,col1,row2,col2)

### Bộ test

| Nhóm | Ca thử | Vì sao đáng thử |
|---|---|---|
| 1 | Example 1 qua `run_commands` | khớp đúng format LeetCode |
| 2 | gán ô `(2, 1)` (đang là 2) thành 5 | bắt lỗi "cộng thay vì gán" — Example 1 không bắt được |
| 3 | update **cùng một ô** nhiều lần; gán lại đúng giá trị cũ | bắt lỗi quên cập nhật `self.nums`; nhánh `delta == 0` |
| 4 | ma trận `1 × 1`; một hàng `1 × n`; một cột `m × 1` | biên nhỏ nhất, một chiều bằng 1 |
| 5 | số âm; vùng là cả ma trận; vùng là 1 ô; vùng chạm cạnh trên/trái | `prefix(0, ...) = 0` phải đúng |
| 6 | cây build tuyến tính **giống hệt** cây build bằng `_add` | kiểm chứng mẹo build 2 lượt |
| 7 | ma trận `200 × 200` toàn `1000` | kích thước và giá trị lớn nhất: tổng `4 · 10^7` |
| 8 | ma trận gốc `EXAMPLE_MATRIX` không bị sửa | class phải làm việc trên bản sao |

In [26]:
# 1. Example 1
assert run_commands(NumMatrix, EX1_OPS, EX1_ARGS) == EX1_EXPECTED

# 2. Bắt lỗi "cộng thay vì gán": ô (2,1) đang là 2, gán thành 5 -> tổng vùng tăng 3
ops4 = ["NumMatrix", "sumRegion", "update", "sumRegion"]
assert run_commands(NumMatrix, ops4, [[EXAMPLE_MATRIX], [2, 1, 4, 3], [2, 1, 5], [2, 1, 4, 3]]) == [None, 8, None, 11]

# 3. update cùng một ô nhiều lần + gán lại đúng giá trị cũ
obj = NumMatrix([[1, 2], [3, 4]])
obj.update(0, 0, 10); obj.update(0, 0, 10); obj.update(0, 0, -5)
assert obj.sumRegion(0, 0, 1, 1) == -5 + 2 + 3 + 4
obj.update(1, 1, 4)                                   # delta = 0
assert obj.sumRegion(1, 1, 1, 1) == 4

# 4. 1x1, 1xn, mx1
one = NumMatrix([[7]])
assert one.sumRegion(0, 0, 0, 0) == 7
one.update(0, 0, -1000)
assert one.sumRegion(0, 0, 0, 0) == -1000
row = NumMatrix([[1, 2, 3, 4, 5]])
assert row.sumRegion(0, 1, 0, 3) == 9
row.update(0, 2, 0)
assert row.sumRegion(0, 0, 0, 4) == 12
col = NumMatrix([[1], [2], [3], [4]])
assert col.sumRegion(1, 0, 3, 0) == 9
col.update(3, 0, -4)
assert col.sumRegion(0, 0, 3, 0) == 2

# 5. số âm, cả ma trận, một ô, vùng chạm cạnh trên/trái
neg = NumMatrix([[-1, -2, -3], [-4, -5, -6]])
assert neg.sumRegion(0, 0, 1, 2) == -21
assert neg.sumRegion(1, 1, 1, 1) == -5
assert neg.sumRegion(0, 0, 0, 0) == -1
neg.update(0, 0, 100)
assert neg.sumRegion(0, 0, 1, 0) == 96

# 6. build tuyến tính == build bằng _add
assert NumMatrix(EXAMPLE_MATRIX).tree == NumMatrixFenwick2D(EXAMPLE_MATRIX).tree

# 7. kích thước và giá trị lớn nhất (dấu _ trong số chỉ để dễ đọc)
big = NumMatrix([[1000] * 200 for _ in range(200)])
assert big.sumRegion(0, 0, 199, 199) == 40_000_000
big.update(199, 199, -1000)
assert big.sumRegion(199, 199, 199, 199) == -1000
assert big.sumRegion(0, 0, 199, 199) == 40_000_000 - 2000

# 8. ma trận gốc không bị sửa
assert EXAMPLE_MATRIX[3][2] == 0 and EXAMPLE_MATRIX[2][1] == 2
print("Tất cả test của NumMatrix đều PASS!")

Tất cả test của NumMatrix đều PASS!


### Đối chiếu chéo 6 cài đặt trên chuỗi lệnh ngẫu nhiên (`random.seed` cố định)

Sinh 500 chuỗi lệnh kiểu LeetCode: ma trận ngẫu nhiên tới `6 × 6`, giá trị trong `[-1000, 1000]`, 1–25 lệnh `update`/`sumRegion` xen kẽ. Giá trị `val` của `update` đôi khi được chọn từ một tập nhỏ (`-1000, -1, 0, 1, 1000`) để thường xuyên có cảnh **gán lại cùng giá trị** hoặc **gán vào cùng ô nhiều lần**. Cài đặt ngây thơ (cách 1) đóng vai "trọng tài"; mọi cài đặt còn lại phải cho **output giống hệt**. Sau đó chạy thêm 30 chuỗi dài hơn (ma trận tới `30 × 30`, 300 lệnh).

In [27]:
IMPLEMENTATIONS = [NumMatrixNaive, NumMatrixPrefix2D, NumMatrixRowPrefix,
                   NumMatrixFenwick2D, NumMatrixRowFenwick, NumMatrix]

def random_commands(max_size, max_ops):
    m, n = random.randint(1, max_size), random.randint(1, max_size)
    matrix = [[random.randint(-1000, 1000) for _ in range(n)] for _ in range(m)]
    ops, args = ["NumMatrix"], [[matrix]]
    for _ in range(random.randint(1, max_ops)):
        if random.random() < 0.5:
            val = random.choice([-1000, -1, 0, 1, 1000, random.randint(-1000, 1000)])
            ops.append("update")
            args.append([random.randrange(m), random.randrange(n), val])
        else:
            r1, r2 = sorted((random.randrange(m), random.randrange(m)))
            c1, c2 = sorted((random.randrange(n), random.randrange(n)))
            ops.append("sumRegion")
            args.append([r1, c1, r2, c2])
    return ops, args

random.seed(308)
total_ops = 0
for max_size, max_ops, trials in [(6, 25, 500), (30, 300, 30)]:
    for _ in range(trials):
        ops, args = random_commands(max_size, max_ops)
        expected = run_commands(NumMatrixNaive, ops, args)
        for cls in IMPLEMENTATIONS[1:]:
            got = run_commands(cls, ops, args)
            assert got == expected, (cls.__name__, ops, args, got, expected)
        total_ops += len(ops) - 1
print(f"530 chuỗi lệnh ngẫu nhiên ({total_ops} lệnh): cả {len(IMPLEMENTATIONS)} cài đặt cho output giống hệt nhau.")

530 chuỗi lệnh ngẫu nhiên (11598 lệnh): cả 6 cài đặt cho output giống hệt nhau.


## 16. Đếm bước và đo thời gian: lý thuyết có khớp thực tế?

**Đếm bước (tất định, không phụ thuộc máy).** Với Fenwick, số bước của `add(i)` là độ dài đường nhảy `i, i + lowbit(i), ...` (lớn nhất khi `i = 1`: `1, 2, 4, 8, ...`); số bước của `prefix(i)` là **số bit 1** của `i` (`bin(i).count("1")`). Cell dưới lấy **giá trị lớn nhất** của các đại lượng đó cho `m = n = 200` (đề) và `m = n = 2000` (giả định lớn hơn để thấy xu hướng), rồi lập bảng "số ô/giá trị phải đọc hoặc sửa trong trường hợp xấu nhất" của mỗi cách.

**Đo thời gian (phụ thuộc máy).** Phần sau của cùng cell dựng ma trận `200 × 200` và một chuỗi lệnh **bất lợi**: xen kẽ `update(0, 0, ...)` (ô góc — tệ nhất cho cách 2 và cách 3) và `sumRegion` cả ma trận (tệ nhất cho cách 1 và cách 3). Cách 1 và 2 chỉ chạy 200 lệnh rồi nhân lên để **ước tính** cho 5000 lệnh (chạy đủ thì quá lâu); các cách còn lại chạy đủ 5000 lệnh. Con số cụ thể tùy máy — điều đáng xem là **độ lớn chênh lệch**.

In [28]:
def update_path_len(i, n):
    steps = 0
    while i <= n:
        steps += 1
        i += i & -i
    return steps

for N in [200, 2000]:
    up = max(update_path_len(i, N) for i in range(1, N + 1))      # bước lớn nhất của add
    qp = max(bin(i).count("1") for i in range(1, N + 1))          # bước lớn nhất của prefix
    print(f"m = n = {N}:  add tối đa {up} bước/chiều, prefix tối đa {qp} bước/chiều")
    rows = [
        ("1. Ngây thơ",         1,         N * N),
        ("2. Prefix 2D",        N * N,     4),
        ("3. Prefix theo hàng", N,         2 * N),
        ("4. Fenwick 2D",       up * up,   4 * qp * qp),
        ("5a. Fenwick mỗi hàng", up,       N * 2 * qp),
    ]
    for name, u, s in rows:
        print(f"   {name:<21} update: {u:>9,}   sumRegion: {s:>9,}")

import time

def adversarial_workload(m, n, q):
    ops = []
    for k in range(q):
        if k % 2 == 0:
            ops.append(("update", [0, 0, (k % 2001) - 1000]))        # ô góc, giá trị luôn đổi
        else:
            ops.append(("sumRegion", [0, 0, m - 1, n - 1]))          # cả ma trận
    return ops

def run_timed(cls, matrix, ops):
    start = time.perf_counter()
    obj = cls(matrix)
    results = [getattr(obj, name)(*a) for name, a in ops]
    return time.perf_counter() - start, [x for x in results if x is not None]

random.seed(2026)
mat200 = [[random.randint(-1000, 1000) for _ in range(200)] for _ in range(200)]
work = adversarial_workload(200, 200, 5000)
reference = None
for cls in IMPLEMENTATIONS:
    q = 200 if cls in (NumMatrixNaive, NumMatrixPrefix2D) else 5000
    t, res = run_timed(cls, mat200, work[:q])
    if reference is None:
        reference = res                                              # kết quả của cách 1 (200 lệnh)
    k = min(len(res), len(reference))
    assert res[:k] == reference[:k]                                  # mọi cách cho cùng kết quả
    note = f"(chạy {q} lệnh, ước tính 5000 lệnh ≈ {t * 5000 / q:6.2f} s)" if q < 5000 else "(chạy đủ 5000 lệnh)"
    print(f"{cls.__name__:<20} {t:7.3f} s  {note}")

m = n = 200:  add tối đa 8 bước/chiều, prefix tối đa 7 bước/chiều
   1. Ngây thơ           update:         1   sumRegion:    40,000
   2. Prefix 2D          update:    40,000   sumRegion:         4
   3. Prefix theo hàng   update:       200   sumRegion:       400
   4. Fenwick 2D         update:        64   sumRegion:       196
   5a. Fenwick mỗi hàng  update:         8   sumRegion:     2,800
m = n = 2000:  add tối đa 11 bước/chiều, prefix tối đa 10 bước/chiều
   1. Ngây thơ           update:         1   sumRegion: 4,000,000
   2. Prefix 2D          update: 4,000,000   sumRegion:         4
   3. Prefix theo hàng   update:     2,000   sumRegion:     4,000
   4. Fenwick 2D         update:       121   sumRegion:       400
   5a. Fenwick mỗi hàng  update:        11   sumRegion:    40,000


NumMatrixNaive         0.121 s  (chạy 200 lệnh, ước tính 5000 lệnh ≈   3.02 s)


NumMatrixPrefix2D      0.155 s  (chạy 200 lệnh, ước tính 5000 lệnh ≈   3.87 s)
NumMatrixRowPrefix     0.050 s  (chạy đủ 5000 lệnh)


NumMatrixFenwick2D     0.091 s  (chạy đủ 5000 lệnh)


NumMatrixRowFenwick    0.150 s  (chạy đủ 5000 lệnh)
NumMatrix              0.027 s  (chạy đủ 5000 lệnh)


## 17. Vì sao cách chính đúng? (invariant)

**Bất biến (invariant)** — điều luôn đúng sau `__init__` và sau mỗi `update`, với **mọi** `i, j`:

```text
tree[i][j] = tổng các giá trị HIỆN TẠI trong khối (hàng i-lowbit(i)+1..i) × (cột j-lowbit(j)+1..j)
nums[r][c] = giá trị HIỆN TẠI của ô (r, c)
```

1. **Khởi tạo đúng**: cách đơn giản cộng từng ô vào đúng các khối chứa nó; cách tuyến tính gom đúng các khối con (mục 11.6, làm lần lượt theo từng chiều).
2. **`update` giữ bất biến**: giá trị ô `(row, col)` đổi đúng `delta = val - nums[row][col]`. Khối nào **chứa** ô đó thì tổng khối đổi đúng `delta`; khối khác không đổi. Hai vòng nhảy `+= lowbit` liệt kê **đúng và đủ** các khối chứa ô (tích của hai đường nhảy 1D, mục 11.5 và 12.2). Gán `nums[row][col] = val` giữ nửa sau của bất biến — nhờ đó lần `update` sau tính `delta` đúng.
3. **`prefix(i, j)` đúng**: hai vòng nhảy `-= lowbit` chia khối `1..i × 1..j` thành các khối `tree` **rời nhau và phủ kín** (tích của hai phép chia 1D, mục 11.4). Cộng chúng lại theo bất biến ra đúng tổng.
4. **`sumRegion` đúng**: bao hàm – loại trừ trên 4 tiền tố, giống hệt #304: mỗi ô trong vùng được tính đúng 1 lần (chỉ nằm trong A); ô nằm trong B hoặc C nhưng không trong D bị cộng 1 lần (A) rồi trừ 1 lần; ô trong D cộng 2 lần (A, D) trừ 2 lần (B, C) — tổng đóng góp bằng 0.

Cell dưới kiểm tra bất biến bằng **vét cạn**: sau `__init__` và sau mỗi `update` ngẫu nhiên, tính lại **mọi** `tree[i][j]` theo định nghĩa và so với cây thật của `NumMatrix`.

In [29]:
def check_invariant(obj):
    for i in range(1, obj.m + 1):
        for j in range(1, obj.n + 1):
            a, b, c, d = block_of(i, j)
            expect = sum(obj.nums[r - 1][cc - 1] for r in range(a, b + 1) for cc in range(c, d + 1))
            assert obj.tree[i][j] == expect, (i, j, obj.tree[i][j], expect)

random.seed(7)
checked = 0
for _ in range(100):
    m, n = random.randint(1, 9), random.randint(1, 9)
    mat = [[random.randint(-9, 9) for _ in range(n)] for _ in range(m)]
    obj = NumMatrix(mat)
    check_invariant(obj)
    for _ in range(10):
        obj.update(random.randrange(m), random.randrange(n), random.randint(-9, 9))
        check_invariant(obj)
        checked += 1
print(f"Bất biến 'tree[i][j] = tổng khối' đúng ngay sau __init__ (100 ma trận) và sau {checked} lần update ngẫu nhiên.")

Bất biến 'tree[i][j] = tổng khối' đúng ngay sau __init__ (100 ma trận) và sau 1000 lần update ngẫu nhiên.


## 18. Độ phức tạp

Gọi `m` = số hàng, `n` = số cột, `log` = log cơ số 2.

| Thao tác (cách chính) | Thời gian | Giải thích |
|---|---|---|
| `__init__` (bản nộp) | `O(mn)` | chép ma trận + 2 lượt đẩy, mỗi ô `O(1)` |
| `__init__` (bản demo) | `O(mn · log m · log n)` | `mn` lần `_add` |
| `update` | `O(log m · log n)` | ≤ `log2(m) + 1` hàng × ≤ `log2(n) + 1` cột |
| `sumRegion` | `O(log m · log n)` | 4 lần `prefix`, mỗi lần ≤ (số bit 1 của `i`) × (số bit 1 của `j`) |
| Bộ nhớ | `O(mn)` | `tree` cỡ `(m+1)(n+1)` + `nums` cỡ `m·n` |

Với `m = n = 200` (mục 16): `update` chạm tối đa `8 × 8 = 64` ô `tree`, mỗi `prefix` tối đa `7 × 7 = 49` ô. Toàn bộ `5000` lệnh chỉ cỡ `10^6` bước.

## 19. Bảng so sánh các cách

| Cách | Ý tưởng | `__init__` | `update` | `sumRegion` | Bộ nhớ thêm | Nhận xét |
|---|---|---|---|---|---|---|
| 1. Ngây thơ | lưu ma trận, cộng từng ô | `O(mn)` | `O(1)` | `O(mn)` | `O(mn)` (bản sao) | "trọng tài"; TLE khi hỏi nhiều vùng lớn |
| 2. Prefix 2D (#304) | `P[i][j]` = tổng góc trên-trái | `O(mn)` | `O(mn)` | `O(1)` | `O(mn)` | TLE khi update nhiều (nhất là gần góc trên-trái) |
| 3. Prefix theo hàng | mỗi hàng một prefix 1D | `O(mn)` | `O(n)` | `O(m)` | `O(mn)` | cân bằng, dễ viết, đủ cho `200 × 200` |
| **4. Fenwick 2D** | khối lũy thừa 2 theo cả 2 chiều | `O(mn)` | `O(log m · log n)` | `O(log m · log n)` | `O(mn)` | tốt nhất về Big-O, code ngắn — **cách chính** |
| 5a. Fenwick mỗi hàng | `m` cây Fenwick 1D | `O(mn)` | `O(log n)` | `O(m log n)` | `O(mn)` | dùng lại Fenwick 1D; lệch về "hỏi đắt" |
| 5b. Segment tree 2D | cây (hàng) của cây (cột) | `O(mn)` | `O(log m · log n)` | `O(log m · log n)` | cỡ `16mn` | linh hoạt (min/max, cập nhật đoạn), code dài |

Lộ trình tư duy: **cộng từng ô** (1) → **tính sẵn mọi tiền tố** (2) → **tính sẵn từng hàng** để chia đều chi phí (3) → **tính sẵn các khối lũy thừa 2** để cả hai thao tác chỉ còn `log` (4).

## 20. Lỗi thường gặp

### Lỗi 1: Coi `update` là "cộng thêm `val`"

```python
def update(self, row, col, val):
    self._add(row + 1, col + 1, val)        # SAI: cộng val, trong khi đề yêu cầu GÁN
```

Example 1 **không** bắt được lỗi này (ô `(3, 2)` đang là 0 nên "cộng 2" và "gán 2" trùng nhau). Sửa: `delta = val - self.nums[row][col]`.

### Lỗi 2: Tính `delta` nhưng quên gán `self.nums[row][col] = val`

Lần update đầu vẫn đúng, nhưng lần update **thứ hai vào cùng ô** sẽ tính `delta` so với giá trị **quá cũ** → cộng lặp. Ví dụ ô đang là 2, gán thành 5 hai lần liên tiếp: đúng ra lần hai `delta = 0`, nhưng code lỗi tính lại `delta = 5 - 2 = 3`.

Cell dưới tạo hai class lỗi bằng **kế thừa** (inheritance): `class BugAddVal(NumMatrixFenwick2D)` dùng lại toàn bộ code của class cha, chỉ **viết đè** method `update`. Chuỗi lệnh: hỏi vùng → gán ô `(2, 1)` thành 5 → hỏi → gán ô `(2, 1)` thành 5 lần nữa → hỏi.

In [30]:
class BugAddVal(NumMatrixFenwick2D):
    def update(self, row, col, val):
        self._add(row + 1, col + 1, val)                 # Lỗi 1: cộng val

class BugForgetNums(NumMatrixFenwick2D):
    def update(self, row, col, val):
        delta = val - self.nums[row][col]                # Lỗi 2: thiếu self.nums[row][col] = val
        self._add(row + 1, col + 1, delta)

ops6 = ["NumMatrix", "sumRegion", "update", "sumRegion", "update", "sumRegion"]
args6 = [[EXAMPLE_MATRIX], [2, 1, 4, 3], [2, 1, 5], [2, 1, 4, 3], [2, 1, 5], [2, 1, 4, 3]]
print("Đúng                 :", run_commands(NumMatrixFenwick2D, ops6, args6))
print("Lỗi 1 (cộng val)     :", run_commands(BugAddVal, ops6, args6))
print("Lỗi 2 (quên nums)    :", run_commands(BugForgetNums, ops6, args6))
print("Example 1 với Lỗi 1  :", run_commands(BugAddVal, EX1_OPS, EX1_ARGS), "<- vẫn 'đúng', nên dễ bị lừa!")

Đúng                 : [None, 8, None, 11, None, 11]
Lỗi 1 (cộng val)     : [None, 8, None, 13, None, 18]
Lỗi 2 (quên nums)    : [None, 8, None, 11, None, 14]
Example 1 với Lỗi 1  : [None, 8, None, 10] <- vẫn 'đúng', nên dễ bị lừa!


### Lỗi 3: Dùng chỉ số 0 trong Fenwick → vòng lặp vô hạn

`lowbit(0) = 0 & -0 = 0`, nên `i += lowbit(i)` với `i = 0` **không bao giờ tăng** → chương trình treo. Ngược lại, `while i > 0` trong `prefix` với `i = 0` thì không chạy vòng nào — trả 0, **sai trong im lặng** nếu bạn quên `+1`. Quy tắc: **mọi** chỉ số đi vào cây đều `+1`.

Cell dưới mô phỏng `add` với một "cầu chì" giới hạn số bước để không bị treo thật.

In [31]:
def add_with_fuse(tree, i, delta, max_steps=6):
    n = len(tree) - 1
    visited = []
    while i <= n and len(visited) < max_steps:      # cầu chì: tối đa max_steps bước
        tree[i] += delta
        visited.append(i)
        i += i & -i
    return visited

print("0 & -0 =", 0 & -0)
print("add bắt đầu từ i = 0 (SAI) :", add_with_fuse([0] * 9, 0, 5), "... không bao giờ thoát nếu không có cầu chì")
print("add bắt đầu từ i = 1 (ĐÚNG):", add_with_fuse([0] * 9, 1, 5))

0 & -0 = 0
add bắt đầu từ i = 0 (SAI) : [0, 0, 0, 0, 0, 0] ... không bao giờ thoát nếu không có cầu chì
add bắt đầu từ i = 1 (ĐÚNG): [1, 2, 4, 8]


### Lỗi 4: Tạo bảng 2D bằng `[[0] * (n + 1)] * (m + 1)`

Phép `*` trên list **nhân bản tham chiếu**: mọi hàng là **cùng một list** trong bộ nhớ; sửa `tree[1][1]` thì "mọi hàng" đều đổi theo. Phải dùng list comprehension `[[0] * (n + 1) for _ in range(m + 1)]` để mỗi vòng tạo một list mới. (Bên trong một hàng, `[0] * (n + 1)` thì không sao, vì số nguyên không bị sửa tại chỗ.)

### Lỗi 5: Lệch 1 trong công thức bao hàm – loại trừ

Viết `prefix(row1 + 1, ...)` thay vì `prefix(row1, ...)` sẽ trừ luôn hàng `row1` — vốn thuộc vùng cần tính. Quy tắc: góc "**trước** vùng" dùng `row1`, `col1` (không `+1`); góc "**hết** vùng" dùng `row2 + 1`, `col2 + 1`. Với Example 1, công thức lệch 1 ở cả hai góc "trước vùng" tính ra tổng của hàng `3..4` × cột `2..3` thay vì hàng `2..4` × cột `1..3`.

In [32]:
bad = [[0] * 3] * 3
bad[1][1] = 7
print("Lỗi 4 - nhân bản tham chiếu:", bad)
good = [[0] * 3 for _ in range(3)]
good[1][1] = 7
print("Đúng - list comprehension  :", good)

obj = NumMatrixFenwick2D(EXAMPLE_MATRIX)
r1, c1, r2, c2 = 2, 1, 4, 3
wrong = (obj._prefix(r2 + 1, c2 + 1) - obj._prefix(r1 + 1, c2 + 1)
         - obj._prefix(r2 + 1, c1 + 1) + obj._prefix(r1 + 1, c1 + 1))
print("Lỗi 5 - công thức lệch 1:", wrong, "| đúng:", obj.sumRegion(r1, c1, r2, c2))

Lỗi 4 - nhân bản tham chiếu: [[0, 7, 0], [0, 7, 0], [0, 7, 0]]
Đúng - list comprehension  : [[0, 0, 0], [0, 7, 0], [0, 0, 0]]
Lỗi 5 - công thức lệch 1: 4 | đúng: 8


### Lỗi 6: Các lỗi "nhỏ mà chết"

- **Dùng chung biến cho vòng trong** của Fenwick 2D (`while j <= n: ... j += lowbit(j)` mà không khởi tạo lại `j` cho mỗi hàng) → từ hàng thứ hai, vòng trong không chạy hoặc chạy sai chỗ. Luôn dùng `jj = j` mới.
- **Điều kiện dừng sai**: `while i < n` thay vì `while i <= n` khi sửa → bỏ sót `tree[n]` (khối lớn nhất, thường chứa nhiều ô nhất).
- **Rebuild cả bảng prefix sau mỗi update** (biến thể tệ hơn của cách 2) → `O(mn)` mỗi lần, dễ TLE.
- **Sửa thẳng `matrix` đầu vào** mà không copy → nếu người gọi dùng lại ma trận đó (như các test trong notebook), kết quả các test sau bị sai lây.

## 21. Mẹo nhận dạng pattern

- Đề có **hai loại thao tác xen kẽ**: "sửa một phần tử" + "hỏi tổng một đoạn/vùng" → nghĩ ngay tới **Fenwick Tree** (hoặc segment tree).
- Chỉ hỏi, **không sửa** → prefix sum (#303, #304) là đủ, `O(1)` mỗi câu hỏi.
- **Sửa theo đoạn**, **hỏi từng điểm** → mảng hiệu (difference array) + Fenwick (Bài 5).
- Phép gộp có **phép ngược** (tổng, XOR) → Fenwick; không có (min, max, gcd) → segment tree.
- Kích thước nhỏ (vài trăm) → prefix theo hàng (cách 3) cũng đủ; muốn chắc chắn / kích thước lớn → Fenwick 2D.
- Thấy chữ **"update ... to be val"** → nhớ ngay `delta = val - old` và giữ mảng giá trị hiện tại.
- Đếm "có bao nhiêu phần tử nhỏ hơn `x` đã gặp" khi duyệt mảng → Fenwick trên **trục giá trị** (Bài 6, #315).

## 22. Biến thể & bài liên quan

| Bài | Nội dung | Kỹ thuật liên hệ |
|---|---|---|
| #303 Range Sum Query - Immutable | 1D, không sửa | prefix sum 1D |
| #304 Range Sum Query 2D - Immutable | 2D, không sửa | prefix sum 2D, bao hàm – loại trừ |
| #307 Range Sum Query - Mutable | 1D, có update | Fenwick 1D / segment tree — bản 1D của bài này |
| #315 Count of Smaller Numbers After Self | đếm phần tử nhỏ hơn ở bên phải | Fenwick trên trục giá trị (Bài 6) |
| #327 Count of Range Sum | đếm đoạn có tổng trong `[lower, upper]` | prefix sum + Fenwick / merge sort |
| #493 Reverse Pairs | đếm cặp `nums[i] > 2 · nums[j]` | Fenwick / merge sort |
| #1314 Matrix Block Sum | tổng khối quanh mỗi ô | prefix sum 2D |
| #2536 Increment Submatrices by One | cộng 1 vào nhiều hình chữ nhật | mảng hiệu 2D (phép "ngược" của prefix sum 2D) |

Hai biến thể tự nhiên của chính bài này:

- **Cập nhật kiểu "cộng thêm"** (`add(row, col, delta)`) → bỏ hẳn `self.nums`, gọi thẳng `_add`.
- **Cộng một hằng số vào cả hình chữ nhật, hỏi giá trị từng ô** → Fenwick 2D trên **mảng hiệu 2D**: cộng vào 4 góc `(r1, c1) +d`, `(r1, c2+1) −d`, `(r2+1, c1) −d`, `(r2+1, c2+1) +d`; giá trị ô = `prefix(r+1, c+1)`. (Bài 5 là bản 1D của ý này.)

## 23. Bài tập vận dụng

Làm theo thứ tự. Mỗi bài có gợi ý 3 mức ở mục 24 và đáp án ở mục 25 (cuối notebook). Các cell TODO chỉ **định nghĩa** hàm/class — test để ở dạng comment, tự làm xong mới bỏ comment.

### Bài 1 — Tính tay (không chạy code)

Với Fenwick 1D `n = 16`:

1. Tính `lowbit(12)`, `lowbit(10)`, `lowbit(7)`, `lowbit(16)`.
2. `tree[12]`, `tree[10]`, `tree[7]`, `tree[16]` phụ trách đoạn nào?
3. Liệt kê đường nhảy của `prefix(13)` và các đoạn được cộng.
4. Liệt kê đường nhảy của `add(5, delta)`.
5. Fenwick 2D kích thước `8 × 8`: `add` tại ô 1-index `(3, 6)` chạm vào những ô `tree` nào? Bao nhiêu ô?

### Bài 2 — Đoán output

```python
ops  = ["NumMatrix", "sumRegion", "update", "sumRegion", "update", "sumRegion", "update", "sumRegion"]
args = [[[[1, 2], [3, 4]]], [0, 0, 1, 1], [0, 1, 5], [0, 1, 1, 1], [0, 1, 5], [0, 0, 0, 1], [1, 0, -3], [0, 0, 1, 1]]
```

a) Đoán `run_commands(NumMatrix, ops, args)`.

b) Nếu chạy bằng một cài đặt mắc **Lỗi 1** (cộng `val` thay vì gán, như `BugAddVal`) thì output là gì?

### Bài 3 — Điền TODO: Fenwick 1D

Hoàn thiện `add` (đi lên) và `prefix` (đi xuống) của class `ExFenwick` (chỉ số 1-index, cây bắt đầu toàn 0).

In [33]:
class ExFenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def add(self, i, delta):          # i: 1-index
        # TODO: lặp khi i <= n: cộng delta vào tree[i], rồi nhảy i lên khối cha
        raise NotImplementedError

    def prefix(self, i):              # tổng a[1..i]
        # TODO: lặp khi i > 0: cộng tree[i], rồi nhảy i xuống
        raise NotImplementedError

# Test sau khi làm xong:
# f = ExFenwick(8)
# for k, v in enumerate([3, 1, 4, 1, 5, 9, 2, 6], start=1):
#     f.add(k, v)
# assert f.tree[1:] == [3, 4, 4, 9, 5, 14, 2, 31]
# assert f.prefix(7) == 25 and f.prefix(0) == 0
# f.add(3, 5)
# assert f.prefix(3) == 13 and f.prefix(8) == 36

### Bài 4 — Tự viết lại cách chính

Không nhìn code mục 15, viết class `ExNumMatrix` (Fenwick 2D) với đúng 3 method của đề: `__init__(matrix)`, `update(row, col, val)`, `sumRegion(row1, col1, row2, col2)`. Được phép dùng cách build đơn giản. Test bằng `run_commands` với Example 1 và chuỗi lệnh của Bài 2.

In [34]:
class ExNumMatrix:
    def __init__(self, matrix):
        # TODO 1: lưu m, n; self.nums (bản sao); self.tree kích thước (m+1) x (n+1)
        # TODO 2: build cây (gợi ý: nums toàn 0 rồi gọi update cho từng ô)
        raise NotImplementedError

    def update(self, row, col, val):
        # TODO 3: delta = val - giá trị cũ; gán; hai vòng nhảy += lowbit (nhớ +1 chỉ số)
        raise NotImplementedError

    def sumRegion(self, row1, col1, row2, col2):
        # TODO 4: 4 lần prefix với bao hàm - loại trừ
        raise NotImplementedError

# Test sau khi làm xong:
# assert run_commands(ExNumMatrix, EX1_OPS, EX1_ARGS) == [None, 8, None, 10]
# ops_b2  = ["NumMatrix", "sumRegion", "update", "sumRegion", "update", "sumRegion", "update", "sumRegion"]
# args_b2 = [[[[1, 2], [3, 4]]], [0, 0, 1, 1], [0, 1, 5], [0, 1, 1, 1], [0, 1, 5], [0, 0, 0, 1], [1, 0, -3], [0, 0, 1, 1]]
# assert run_commands(ExNumMatrix, ops_b2, args_b2) == run_commands(NumMatrixNaive, ops_b2, args_b2)

### Bài 5 — Biến thể: cộng theo đoạn, hỏi từng điểm (1D)

Viết class `RangeAddArray(nums)`:

- `add_range(left, right, delta)`: cộng `delta` vào **mọi** `nums[left..right]` (0-index, gồm hai đầu);
- `get(index)`: trả giá trị hiện tại của `nums[index]`.

Yêu cầu: cả hai thao tác `O(log n)` (vòng `for` cộng từng phần tử là `O(n)` — không đạt).

Ví dụ: `nums = [1, 2, 3, 4, 5]`; `add_range(1, 3, 10)` → `[1, 12, 13, 14, 5]`; `add_range(0, 1, -1)` → `[0, 11, 13, 14, 5]`.

In [35]:
class RangeAddArray:
    def __init__(self, nums):
        # TODO: lưu bản sao nums và một Fenwick (toàn 0) trên "mảng hiệu" của phần được cộng thêm
        raise NotImplementedError

    def add_range(self, left, right, delta):
        # TODO: chỉ 2 lần add vào Fenwick
        raise NotImplementedError

    def get(self, index):
        # TODO: giá trị gốc + một lần prefix
        raise NotImplementedError

# Test sau khi làm xong:
# ra = RangeAddArray([1, 2, 3, 4, 5])
# ra.add_range(1, 3, 10)
# assert [ra.get(i) for i in range(5)] == [1, 12, 13, 14, 5]
# ra.add_range(0, 1, -1)
# assert [ra.get(i) for i in range(5)] == [0, 11, 13, 14, 5]
# ra.add_range(4, 4, 100)
# assert ra.get(4) == 105

### Bài 6 — Mini challenge: #315 Count of Smaller Numbers After Self

Cho mảng `nums`, trả về list `counts` với `counts[i]` = số phần tử nằm **bên phải** `nums[i]` và **nhỏ hơn** `nums[i]`. Constraints của #315: `1 <= nums.length <= 10^5`, `-10^4 <= nums[i] <= 10^4`.

Yêu cầu: dùng Fenwick trên **trục giá trị** (mỗi vị trí của cây ứng với một giá trị có thể có), độ phức tạp `O(n log V)` với `V = 2 · 10^4 + 1` giá trị.

Ví dụ: `[5, 2, 6, 1]` → `[2, 1, 1, 0]` (bên phải 5 có 2 và 1 nhỏ hơn; bên phải 2 có 1; bên phải 6 có 1; bên phải 1 không có gì); `[-1]` → `[0]`; `[-1, -1]` → `[0, 0]`.

In [36]:
def count_smaller(nums):
    # TODO 1: OFFSET để giá trị -10^4 ứng với vị trí 1 của cây
    # TODO 2: duyệt nums từ PHẢI sang TRÁI
    # TODO 3: với mỗi v: đếm số giá trị < v đã gặp (một prefix), rồi "thêm" v vào cây (add 1)
    # TODO 4: đảo ngược list kết quả
    raise NotImplementedError

# Test sau khi làm xong:
# assert count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]
# assert count_smaller([-1]) == [0]
# assert count_smaller([-1, -1]) == [0, 0]
# assert count_smaller([10000, -10000, 0]) == [2, 0, 0]

## 24. Gợi ý theo mức

**Bài 1**
- Mức 1: viết số ra nhị phân; `lowbit` là giá trị của bit 1 bên phải nhất.
- Mức 2: đoạn của `tree[i]` là `[i − lowbit(i) + 1 .. i]`.
- Mức 3: `prefix`: liên tục xóa bit 1 thấp nhất (`13 = 1101₂ → 1100₂ → 1000₂ → 0`); `add`: liên tục cộng `lowbit` cho tới khi vượt `n`; 2D: lấy **tích** hai đường nhảy (hàng × cột).

**Bài 2**
- Mức 1: vẽ lại ma trận 2×2 sau mỗi `update`.
- Mức 2: `update` là **gán**; gán lại đúng giá trị đang có thì không đổi gì.
- Mức 3: với Lỗi 1, mỗi `update` **cộng** `val` vào giá trị đang có: ô `(0,1)` đi `2 → 7 → 12`, ô `(1,0)` đi `3 → 0`.

**Bài 3**
- Mức 1: `add` đi **lên** (chỉ số tăng), `prefix` đi **xuống** (chỉ số giảm).
- Mức 2: `while i <= self.n: ...; i += i & -i` và `while i > 0: ...; i -= i & -i`.
- Mức 3: thân vòng `add` chỉ 1 dòng `self.tree[i] += delta`; thân vòng `prefix` là `total += self.tree[i]`, nhớ khởi tạo `total = 0` và `return total`.

**Bài 4**
- Mức 1: cần lưu 4 thứ: `m`, `n`, `nums`, `tree` cỡ `(m+1) × (n+1)`.
- Mức 2: mẹo build ngắn nhất: tạo `self.nums` **toàn 0**, rồi gọi `self.update(r, c, matrix[r][c])` cho từng ô — vì từ 0 lên `matrix[r][c]` thì `delta` đúng bằng `matrix[r][c]`.
- Mức 3: `sumRegion` = `p(row2+1, col2+1) − p(row1, col2+1) − p(row2+1, col1) + p(row1, col1)`; vòng trong dùng biến mới `jj = j`.

**Bài 5**
- Mức 1: nghĩ về **mảng hiệu** `d`: phần được cộng thêm tại `index` = `d[0] + d[1] + ... + d[index]`. Cộng `delta` vào đoạn `[left..right]` chỉ làm đổi **2** phần tử của `d`.
- Mức 2: `d[left] += delta`, `d[right + 1] -= delta`; giá trị tại `index` = `nums_gốc[index] + prefix của d tới index` → Fenwick trên `d`.
- Mức 3: `add(left + 1, delta)`, `add(right + 2, -delta)` (nếu `right + 2 > n` thì vòng `while` tự không chạy — không cần `if`); `get(index) = base[index] + prefix(index + 1)`.

**Bài 6**
- Mức 1: duyệt từ phải sang trái; khi tới `nums[i]`, những số "bên phải" chính là những số **đã duyệt qua**.
- Mức 2: Fenwick đếm số lần mỗi giá trị đã xuất hiện: "thêm `v`" = `add(pos(v), 1)`; "số giá trị nhỏ hơn `v`" = `prefix(pos(v) − 1)`.
- Mức 3: `pos(v) = v + 10^4 + 1` nằm trong `[1, 2·10^4 + 1]`; cây cỡ `2·10^4 + 2`; cuối cùng `return res[::-1]`.

## 25. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

1. `12 = 1100₂` → `lowbit = 4`; `10 = 1010₂` → `2`; `7 = 111₂` → `1`; `16 = 10000₂` → `16`.
2. `tree[12]` → `[9..12]`; `tree[10]` → `[9..10]`; `tree[7]` → `[7..7]`; `tree[16]` → `[1..16]`.
3. `prefix(13)`: `13 → 12 → 8 → 0`, cộng `tree[13]` (`[13..13]`) + `tree[12]` (`[9..12]`) + `tree[8]` (`[1..8]`). `13 = 1101₂` có 3 bit 1 → 3 đoạn, dài `1 + 4 + 8 = 13`.
4. `add(5)`: `5 → 6 → 8 → 16 → 32 > 16` (dừng): sửa `tree[5], tree[6], tree[8], tree[16]`.
5. Đường nhảy hàng `3 → 4 → 8` (tiếp theo 16 > 8), cột `6 → 8` (tiếp theo 16 > 8) → `3 × 2 = 6` ô: `(3,6), (3,8), (4,6), (4,8), (8,6), (8,8)`.

### Đáp án Bài 2

a) `[None, 10, None, 9, None, 6, None, 7]`: ma trận `[[1, 2], [3, 4]]` tổng 10 → gán `(0,1) = 5` → `[[1, 5], [3, 4]]`, cột 1 = `5 + 4 = 9` → gán lại 5, không đổi → hàng 0 = `1 + 5 = 6` → gán `(1,0) = -3` → `[[1, 5], [-3, 4]]`, tổng = 7.

b) `[None, 10, None, 11, None, 13, None, 17]`: ô `(0,1)` thành `2 + 5 = 7` (cột 1 = `7 + 4 = 11`), rồi `7 + 5 = 12` (hàng 0 = `1 + 12 = 13`); ô `(1,0)` thành `3 + (-3) = 0`, tổng = `1 + 12 + 0 + 4 = 17`.

Cell dưới kiểm chứng cả hai bài bằng code.

In [37]:
# Kiểm chứng Bài 1
assert [lowbit(x) for x in [12, 10, 7, 16]] == [4, 2, 1, 16]
assert [(i - lowbit(i) + 1, i) for i in [12, 10, 7, 16]] == [(9, 12), (9, 10), (7, 7), (1, 16)]
path, i = [], 13
while i > 0:
    path.append(i)
    i -= lowbit(i)
assert path == [13, 12, 8]
assert update_path(5, 16) == [5, 6, 8, 16]
cells_2d = [(i, j) for i in update_path(3, 8) for j in update_path(6, 8)]
assert cells_2d == [(3, 6), (3, 8), (4, 6), (4, 8), (8, 6), (8, 8)]
print("Bài 1: OK, ô 2D bị chạm:", cells_2d)

# Kiểm chứng Bài 2
ops_b2  = ["NumMatrix", "sumRegion", "update", "sumRegion", "update", "sumRegion", "update", "sumRegion"]
args_b2 = [[[[1, 2], [3, 4]]], [0, 0, 1, 1], [0, 1, 5], [0, 1, 1, 1], [0, 1, 5], [0, 0, 0, 1], [1, 0, -3], [0, 0, 1, 1]]
assert run_commands(NumMatrix, ops_b2, args_b2) == [None, 10, None, 9, None, 6, None, 7]
assert run_commands(BugAddVal, ops_b2, args_b2) == [None, 10, None, 11, None, 13, None, 17]
print("Bài 2: OK")

Bài 1: OK, ô 2D bị chạm: [(3, 6), (3, 8), (4, 6), (4, 8), (8, 6), (8, 8)]
Bài 2: OK


In [38]:
# Đáp án Bài 3
class AnsFenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def add(self, i, delta):
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i

    def prefix(self, i):
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & -i
        return total

f = AnsFenwick(8)
for k, v in enumerate([3, 1, 4, 1, 5, 9, 2, 6], start=1):
    f.add(k, v)
assert f.tree[1:] == [3, 4, 4, 9, 5, 14, 2, 31]
assert f.prefix(7) == 25 and f.prefix(0) == 0
f.add(3, 5)
assert f.prefix(3) == 13 and f.prefix(8) == 36
print("Bài 3: OK")

# Đáp án Bài 4 (cũng chính là template ở mục 26)
class AnsNumMatrix:
    def __init__(self, matrix):
        self.m, self.n = len(matrix), len(matrix[0])
        self.nums = [[0] * self.n for _ in range(self.m)]
        self.tree = [[0] * (self.n + 1) for _ in range(self.m + 1)]
        for r in range(self.m):
            for c in range(self.n):
                self.update(r, c, matrix[r][c])     # từ 0 lên matrix[r][c]: delta = matrix[r][c]

    def update(self, row, col, val):
        delta = val - self.nums[row][col]           # GÁN -> delta
        self.nums[row][col] = val
        i = row + 1                                 # 1-index
        while i <= self.m:
            j = col + 1                             # khởi tạo lại cho mỗi hàng
            while j <= self.n:
                self.tree[i][j] += delta
                j += j & -j
            i += i & -i

    def _prefix(self, i, j):
        s = 0
        while i > 0:
            jj = j
            while jj > 0:
                s += self.tree[i][jj]
                jj -= jj & -jj
            i -= i & -i
        return s

    def sumRegion(self, r1, c1, r2, c2):
        return (self._prefix(r2 + 1, c2 + 1) - self._prefix(r1, c2 + 1)
                - self._prefix(r2 + 1, c1) + self._prefix(r1, c1))

assert run_commands(AnsNumMatrix, EX1_OPS, EX1_ARGS) == [None, 8, None, 10]
assert run_commands(AnsNumMatrix, ops_b2, args_b2) == run_commands(NumMatrixNaive, ops_b2, args_b2)
random.seed(4)
for _ in range(200):
    ops, args = random_commands(6, 25)
    assert run_commands(AnsNumMatrix, ops, args) == run_commands(NumMatrixNaive, ops, args)
print("Bài 4: OK (Example 1, Bài 2, 200 chuỗi lệnh ngẫu nhiên)")

# Đáp án Bài 5 — mảng hiệu + Fenwick
class AnsRangeAddArray:
    def __init__(self, nums):
        self.base = nums[:]                  # giá trị gốc
        self.n = len(nums)
        self.tree = [0] * (self.n + 1)       # Fenwick trên mảng hiệu d của phần cộng thêm

    def _add(self, i, delta):
        while i <= self.n:                   # i > n thì không làm gì (đúng ý: d[n] không cần)
            self.tree[i] += delta
            i += i & -i

    def _prefix(self, i):
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & -i
        return total

    def add_range(self, left, right, delta):
        self._add(left + 1, delta)           # d[left]    += delta
        self._add(right + 2, -delta)         # d[right+1] -= delta

    def get(self, index):
        return self.base[index] + self._prefix(index + 1)

ra = AnsRangeAddArray([1, 2, 3, 4, 5])
ra.add_range(1, 3, 10)
assert [ra.get(i) for i in range(5)] == [1, 12, 13, 14, 5]
ra.add_range(0, 1, -1)
assert [ra.get(i) for i in range(5)] == [0, 11, 13, 14, 5]
ra.add_range(4, 4, 100)
assert ra.get(4) == 105
random.seed(5)
for _ in range(200):
    arr = [random.randint(-50, 50) for _ in range(random.randint(1, 20))]
    ra = AnsRangeAddArray(arr)
    for _ in range(15):
        l = random.randrange(len(arr)); r = random.randrange(l, len(arr)); d = random.randint(-9, 9)
        for k in range(l, r + 1):
            arr[k] += d
        ra.add_range(l, r, d)
        assert [ra.get(k) for k in range(len(arr))] == arr
print("Bài 5: OK")

# Đáp án Bài 6 — #315 với Fenwick trên trục giá trị
def ans_count_smaller(nums):
    OFFSET = 10**4 + 1                       # v = -10^4 -> vị trí 1; v = 10^4 -> vị trí 2*10^4 + 1
    size = 2 * 10**4 + 1
    tree = [0] * (size + 1)
    res = []
    for v in reversed(nums):                 # từ phải sang trái
        i, cnt = v + OFFSET - 1, 0           # đếm các giá trị < v: prefix(pos(v) - 1)
        while i > 0:
            cnt += tree[i]
            i -= i & -i
        res.append(cnt)
        i = v + OFFSET                       # thêm v vào cây
        while i <= size:
            tree[i] += 1
            i += i & -i
    return res[::-1]

assert ans_count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]
assert ans_count_smaller([-1]) == [0]
assert ans_count_smaller([-1, -1]) == [0, 0]
assert ans_count_smaller([10000, -10000, 0]) == [2, 0, 0]
random.seed(315)
for _ in range(200):
    arr = [random.randint(-10**4, 10**4) if random.random() < 0.5 else random.randint(-3, 3)
           for _ in range(random.randint(1, 30))]
    brute = [sum(arr[j] < arr[i] for j in range(i + 1, len(arr))) for i in range(len(arr))]
    assert ans_count_smaller(arr) == brute
print("Bài 6: OK (ví dụ đề + 200 mảng ngẫu nhiên so với brute force)")

Bài 3: OK
Bài 4: OK (Example 1, Bài 2, 200 chuỗi lệnh ngẫu nhiên)
Bài 5: OK
Bài 6: OK (ví dụ đề + 200 mảng ngẫu nhiên so với brute force)


## 26. Template nên nhớ

```python
class NumMatrix:
    def __init__(self, matrix):
        self.m, self.n = len(matrix), len(matrix[0])
        self.nums = [[0] * self.n for _ in range(self.m)]
        self.tree = [[0] * (self.n + 1) for _ in range(self.m + 1)]
        for r in range(self.m):
            for c in range(self.n):
                self.update(r, c, matrix[r][c])     # build: "gán" từng ô từ 0

    def update(self, row, col, val):
        delta = val - self.nums[row][col]           # GÁN -> delta
        self.nums[row][col] = val
        i = row + 1                                 # vào cây thì +1
        while i <= self.m:
            j = col + 1
            while j <= self.n:
                self.tree[i][j] += delta
                j += j & -j                         # sửa: đi LÊN
            i += i & -i

    def _prefix(self, i, j):
        s = 0
        while i > 0:
            jj = j
            while jj > 0:
                s += self.tree[i][jj]
                jj -= jj & -jj                      # hỏi: đi XUỐNG
            i -= i & -i
        return s

    def sumRegion(self, r1, c1, r2, c2):
        return (self._prefix(r2 + 1, c2 + 1) - self._prefix(r1, c2 + 1)
                - self._prefix(r2 + 1, c1) + self._prefix(r1, c1))
```

(Template này chính là đáp án Bài 4 đã được kiểm chứng ở trên; bản nộp mục 15 khác ở chỗ build tuyến tính `O(mn)`.)

Câu thần chú: **"Gán thì tính delta; vào cây thì +1; sửa thì `+= lowbit`, hỏi thì `-= lowbit`; vùng thì 4 góc A − B − C + D."**

## 27. Checklist trước khi submit

- [ ] `update` tính `delta = val - self.nums[row][col]` **và** gán `self.nums[row][col] = val`.
- [ ] Mọi chỉ số đi vào cây đều `+1` (không bao giờ gọi `add` với 0).
- [ ] Sửa: `while i <= m` / `while j <= n`; hỏi: `while i > 0` / `while j > 0`.
- [ ] Vòng trong dùng biến riêng, **khởi tạo lại** cho mỗi hàng.
- [ ] Bảng 2D tạo bằng list comprehension, không dùng `[[0] * n] * m`.
- [ ] `sumRegion` = `A − B − C + D`: góc "hết vùng" dùng `row2+1, col2+1`; góc "trước vùng" dùng `row1, col1`.
- [ ] Không sửa thẳng `matrix` đầu vào (làm việc trên bản sao).
- [ ] Thử tay: Example 1 → `[null, 8, null, 10]`; thêm một `update` vào ô **khác 0**; `update` hai lần cùng ô; ma trận `1 × 1`.